In [ ]:
import os
from pathlib import Path

# Run this notebook from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().parent
PROCESSING_DIR = PROJECT_ROOT / 'data' / 'processing'
PROCESSING_DIR.mkdir(parents=True, exist_ok=True)
full_data_path = str(PROCESSING_DIR)


# 1. LLM-data extraction

In [ ]:
import os
import time
import json
import pandas as pd
from openai import OpenAI

# ==========================================
# 1. API Configuration
# ==========================================
API_KEY = os.environ["LLM_API_KEY"]

client = OpenAI(
    api_key=API_KEY,
    base_url='https://api.zetatechs.com/v1'
)
model = "qwen-flash"

# ==========================================
# 2. System Prompt (Optimized for JSON)
# ==========================================
SYSTEM_PROMPT = """
You are an expert in Environmental Psychology and Spatial Data Science.
Your task is to perform a granular sentiment analysis on social media "Citywalk" notes.
Extract Points of Interest (POIs) and evaluate them using the PAD (Pleasure-Arousal-Dominance) Emotional State Model.

PAD Dimension Definitions:
1. Pleasure (P): (+) Delicious/beautiful/healing; (-) Disappointing/bad; (0) Neutral; (N/A) No data.
2. Arousal (A): (+) Crowded/exciting/noisy; (-) Quiet/slow-paced/relaxing; (0) Neutral; (N/A) No data.
3. Dominance (D): (+) Free to wander/autonomy; (-) Restricted/crowded/forced; (0) Neutral; (N/A) No data.

Rules:
1. Global Route Inheritance: If the text has a global sentiment for the entire route, apply its PAD values to every POI.
2. Override Rule: If a specific POI has its own unique description, it overrides the global sentiment.
3. The "Empty" Rule: If a POI is just a geographic marker with no emotional descriptors, mark PAD as "N/A".
4. Accuracy: Only summarize based on the provided text. Do not hallucinate.

OUTPUT FORMAT:
You MUST output strictly in JSON format as a list of dictionaries. Do not include any other text or markdown outside the JSON.
Example format:
[
  {
    "location_index": 1,
    "location_POI": "Xian Road Flower Market",
    "text_evidence": "quiet and comfortable, slow pace",
    "is_inherited": "No",
    "P_score": "+",
    "A_score": "-",
    "D_score": "+",
    "reasoning": "Comfortable implies P+; Slow pace/Quiet implies A-..."
  }
]
"""

# ==========================================
# 3. Main Processing Function (支持断点续传版本)
# ==========================================
def process_notes_to_pad(input_csv_path, output_csv_path):
    print(f"Loading data from {input_csv_path}...")
    df = pd.read_csv(input_csv_path, encoding='utf-8-sig', on_bad_lines='skip', engine='python', encoding_errors='replace')

    # 【新增功能 1：读取已处理进度】
    processed_cases = set()
    if os.path.exists(output_csv_path):
        try:
            # 读取已有的结果文件
            existing_df = pd.read_csv(output_csv_path, encoding='utf-8-sig')
            if 'note_index' in existing_df.columns:
                processed_cases = set(existing_df['note_index'].unique())
                print(f"✅ 发现进度存档！已跳过 {len(processed_cases)} 条已处理的数据。")
        except Exception as e:
            print(f"⚠️ 读取进度存档失败 ({e})，将从头开始运行。")

    # 定义统一的列顺序
    columns_order = ['note_index', 'original_date', 'location_index', 'location_POI',
                     'text_evidence', 'is_inherited', 'P_score', 'A_score', 'D_score', 'reasoning']

    # Iterate through each row in your CSV
    for index, row in df.iterrows():
        note_index = f"Case_{index + 1}"
        note_content = row.get('笔记内容', '')

        # 【新增功能 2：跳过已处理的数据】
        if note_index in processed_cases:
            print(f"Skipping {note_index} (Already processed)")
            continue

        # Skip empty content
        if pd.isna(note_content) or str(note_content).strip() == "":
            print(f"Skipping {note_index} due to empty content.")
            continue

        print(f"Processing {note_index}...")
        user_message = f"Note Index: {note_index}\nNote Content: {note_content}"

        row_extracted_data = [] # 临时存放当前这条笔记提取出的数据

        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.1,
            )

            raw_response = response.choices[0].message.content
            clean_json = raw_response.strip().strip('```json').strip('```')
            poi_list = json.loads(clean_json)

            if isinstance(poi_list, dict):
                for val in poi_list.values():
                    if isinstance(val, list):
                        poi_list = val
                        break

            # 组装当前笔记的 POI 数据
            for poi in poi_list:
                poi['note_index'] = note_index
                poi['original_date'] = row.get('笔记发布时间', '')
                row_extracted_data.append(poi)

        except Exception as e:
            print(f"❌ Error processing {note_index}: {e}")
            # 如果报错了，为了防止卡死，我们也记录一个错误占位符，以后可以手动查
            row_extracted_data.append({
                'note_index': note_index,
                'original_date': row.get('笔记发布时间', ''),
                'location_POI': f"ERROR: {e}"
            })

        # 【新增功能 3：避免死循环，如果大模型认为没有地点，写入一个空占位符】
        # 这样下次启动时，依然能识别为“已处理”，避免重复去请求这条没有地点的笔记
        if not row_extracted_data:
            row_extracted_data.append({
                'note_index': note_index,
                'original_date': row.get('笔记发布时间', ''),
                'location_POI': "No POI Extracted"
            })

        # 【新增功能 4：实时追加保存到本地文件 (Incremental Save) 】
        if row_extracted_data:
            temp_df = pd.DataFrame(row_extracted_data)

            # 补齐缺失的列，确保表格结构完整
            for col in columns_order:
                if col not in temp_df.columns:
                    temp_df[col] = ""
            temp_df = temp_df[columns_order]

            # 判断文件是否存在，如果不存在则写入表头 (header=True)，存在则不写表头 (header=False)
            write_header = not os.path.exists(output_csv_path)

            # mode='a' 代表追加模式 (append)
            temp_df.to_csv(output_csv_path, mode='a', index=False, header=write_header, encoding='utf-8-sig')

        # Polite delay to avoid hitting API rate limits
        time.sleep(0.5)

    print(f"\n🎉 所有数据处理完毕！结果保存在: {output_csv_path}")

# ==========================================
# Run the script
# ==========================================
if __name__ == "__main__":
    # Specify your file names here
    INPUT_FILE = r"../data/raw/source_notes_anonymized.csv"
    OUTPUT_FILE = "../data/processing/pad_analysis_results.csv"

    # Uncomment the line below to run the function when your files are ready
    process_notes_to_pad(INPUT_FILE, OUTPUT_FILE)

In [ ]:
import os
import time
import json
import pandas as pd
from openai import OpenAI

# ==========================================
# 1. API Configuration
# ==========================================
API_KEY = os.environ["LLM_API_KEY"]
client = OpenAI(
    api_key=API_KEY,
    base_url='https://api.zetatechs.com/v1'
)
model = "qwen-flash"

# ==========================================
# 2. System Prompt
# ==========================================
SYSTEM_PROMPT = """
You are an expert in Urban Geography and Tourism Behavior.
Your task is to read a user's "City Walk" social media note and categorize the spatial themes of their route.

THEME TAXONOMY:
1. "Historical&Art": Heritage sites, old architecture, temples, Creative zones, museums, indie bookstores.
2. "Commercial&Tourism": Food hunting, cafes, shopping, trendy spots.
3. "Nature": Coastal walks, beaches, parks, mountains.
5. "Local_Life": Community markets, residential alleys, authentic local living.

OUTPUT FORMAT:
You MUST output strictly in JSON format as a single dictionary. Do not include markdown code blocks.
Example:
{
  "Primary_Theme": "Commercial",
  "Theme_Reasoning": "The user primarily focused on cafe hopping and shopping street so it should be Commercial"
}
"""

# ==========================================
# 3. Main Processing Function (Batch Save Version)
# ==========================================
def categorize_notes_robust(input_csv_path, output_csv_path):
    print(f"Loading data from {input_csv_path}...")
    df = pd.read_csv(input_csv_path, encoding='utf-8-sig', on_bad_lines='skip', engine='python', encoding_errors='replace')

    # Checkpoint recovery
    processed_cases = set()
    if os.path.exists(output_csv_path):
        try:
            existing_df = pd.read_csv(output_csv_path, encoding='utf-8-sig')
            if 'note_index' in existing_df.columns:
                processed_cases = set(existing_df['note_index'].unique())
                print(f"✅ 发现进度存档！已安全跳过 {len(processed_cases)} 条已处理的数据。")
        except Exception as e:
            print(f"⚠️ 读取进度存档失败 ({e})，将从头开始运行。")

    columns_order = ['note_index', 'Primary_Theme', 'Theme_Reasoning']

    # 【新增逻辑：批量保存缓存】
    batch_data = []
    SAVE_INTERVAL = 50
    processed_count = 0 # 记录当前批次处理了多少条

    for index, row in df.iterrows():
        note_index = f"Case_{index + 1}"
        note_content = row.get('笔记内容', '')

        if note_index in processed_cases:
            continue
        if pd.isna(note_content) or str(note_content).strip() == "":
            continue

        print(f"Categorizing {note_index}...")
        user_message = f"Note Index: {note_index}\nNote Content: {note_content}"
        extracted_data = {'note_index': note_index}

        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_message}
                ],
                temperature=0.1,
            )

            raw_response = response.choices[0].message.content
            clean_json = raw_response.strip().strip('```json').strip('```').strip()
            theme_data = json.loads(clean_json)

            extracted_data.update({
                'Primary_Theme': theme_data.get('Primary_Theme', 'Unknown'),
                'Theme_Reasoning': theme_data.get('Theme_Reasoning', '')
            })

        except Exception as e:
            print(f"❌ Error parsing {note_index}: {e}")
            extracted_data.update({
                'Primary_Theme': 'ERROR',
                'Theme_Reasoning': f"Parsing Failed: {str(e)}"
            })

        # 【将结果加入缓存列表，而不是直接写入硬盘】
        batch_data.append(extracted_data)
        processed_count += 1

        # 【触发批量保存】
        if len(batch_data) >= SAVE_INTERVAL:
            print(f"💾 达到 {SAVE_INTERVAL} 条，正在批量写入本地文件...")
            temp_df = pd.DataFrame(batch_data)

            for col in columns_order:
                if col not in temp_df.columns:
                    temp_df[col] = ""
            temp_df = temp_df[columns_order]

            write_header = not os.path.exists(output_csv_path)
            temp_df.to_csv(output_csv_path, mode='a', index=False, header=write_header, encoding='utf-8-sig')

            # 清空缓存，准备迎接下一批
            batch_data = []

        time.sleep(0.5)

    # 【收尾逻辑：把不足 50 条的剩余数据写入硬盘】
    if batch_data:
        print(f"💾 正在写入最后剩余的 {len(batch_data)} 条数据...")
        temp_df = pd.DataFrame(batch_data)
        for col in columns_order:
            if col not in temp_df.columns:
                temp_df[col] = ""
        temp_df = temp_df[columns_order]

        write_header = not os.path.exists(output_csv_path)
        temp_df.to_csv(output_csv_path, mode='a', index=False, header=write_header, encoding='utf-8-sig')

    print(f"\n🎉 路线主题分类全部完成！")
    print(f"📄 最终结果已保存至: {output_csv_path}")

if __name__ == "__main__":
    INPUT_FILE = r"../data/raw/source_notes_anonymized.csv"
    OUTPUT_FILE = r"../data/processing/route_cate_results.csv"

    categorize_notes_robust(INPUT_FILE, OUTPUT_FILE)

In [ ]:
import os
import time
import json
import pandas as pd
from openai import OpenAI

# ==========================================
# 1. API Configuration
# ==========================================
API_KEY = os.environ["LLM_API_KEY"]
client = OpenAI(
    api_key=API_KEY,
    base_url='https://api.zetatechs.com/v1'
)
model = "qwen-flash"

# ==========================================
# 2. System Prompt
# ==========================================
SYSTEM_PROMPT = """
You are an expert in Urban Geography specializing in Xiamen, China.
Your task is to judge whether a given location name refers to a SPECIFIC, GEOCODABLE place in Xiamen city.

RULES:
- Return YES if the location is a specific, real place that can be assigned latitude/longitude coordinates in or near Xiamen (e.g. a shop, temple, park, landmark, street, beach, cafe). Some special cases need attention like
internet celebrity locaiton like "做个闲人"，"宇宙有趣研究所"，"张万森的家" which is indeed for a specific location so that is a yes.
- Return NO if the location is:
  * A vague description (e.g. "海边", "巷子", "公园", "某个角落","二楼店铺（已变住户）", "海边树景点", "露营咖啡车","北京","城中村")
  * A non-place (e.g. activity, event, transport route, emotion, food item)
  * A place outside Xiamen or Fujian province
  * A fictional or unclear reference
  * A route/itinerary description

OUTPUT FORMAT:
You MUST output strictly in JSON. No markdown, no explanation outside JSON.
Example:
{"is_geocodable": "YES", "reasoning": "鼓浪屿钢琴博物馆 is a well-known landmark in Xiamen"}
{"is_geocodable": "NO", "reasoning": "海边 is too vague to geocode" or "哈尔滨" is not a place in xiamen}
"""

# ==========================================
# 3. 单条判断函数
# ==========================================
def is_geocodable(poi_name, client, model):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Location name: {poi_name}"}
            ],
            temperature=0.0,
        )
        raw = response.choices[0].message.content.strip().strip('```json').strip('```').strip()
        result = json.loads(raw)
        return result.get('is_geocodable', 'NO').upper() == 'YES'
    except Exception as e:
        print(f"  ⚠️ Error on '{poi_name}': {e}")
        return None  # None = 无法判断，后续单独处理

# ==========================================
# 4. 主处理函数
# ==========================================
CHECKPOINT_CSV = "checkpoint_poi_filter.csv"
SAVE_INTERVAL  = 50

def filter_geocodable_pois(df, poi_column='location_POI'):
    # ── 获取所有唯一POI，减少重复API调用 ──
    unique_pois = df[poi_column].dropna().unique()
    print(f"唯一POI总数: {len(unique_pois)}")

    # ── 断点续传：读取已判断的结果 ──
    if os.path.exists(CHECKPOINT_CSV):
        done_df   = pd.read_csv(CHECKPOINT_CSV, encoding='utf-8-sig')
        done_dict = dict(zip(done_df['location_POI'], done_df['is_geocodable']))
        print(f"✅ 读取断点: 已完成 {len(done_dict)} 个POI")
    else:
        done_dict = {}

    remaining = [p for p in unique_pois if p not in done_dict]
    print(f"剩余待判断: {len(remaining)} 个")

    # ── 主循环 ──
    batch = []
    for i, poi in enumerate(remaining):
        result = is_geocodable(poi, client, model)
        print(f"[{i+1}/{len(remaining)}] 判断: {poi} , 结果: {result}")
        if result is None:
            # API失败时保守保留，避免误删
            done_dict[poi] = True
        else:
            done_dict[poi] = result

        batch.append({'location_POI': poi, 'is_geocodable': done_dict[poi]})
        time.sleep(0.3)

        # 定期保存检查点
        if len(batch) >= SAVE_INTERVAL:
            _save_checkpoint(batch)
            batch = []
            print(f"  💾 检查点已保存")

    # 收尾保存
    if batch:
        _save_checkpoint(batch)
        print(f"  💾 最终检查点已保存")

    # ── 用判断结果过滤原始df ──
    df['_keep'] = df[poi_column].map(lambda x: done_dict.get(x, True))
    df_filtered = df[df['_keep']].drop(columns='_keep').reset_index(drop=True)

    removed = len(df) - len(df_filtered)
    print(f"\n过滤完成 — 原始: {len(df)} 行，保留: {len(df_filtered)} 行，删除: {removed} 行")
    return df_filtered


def _save_checkpoint(batch):
    temp_df = pd.DataFrame(batch)
    write_header = not os.path.exists(CHECKPOINT_CSV)
    temp_df.to_csv(CHECKPOINT_CSV, mode='a', index=False,
                   header=write_header, encoding='utf-8-sig')


# ==========================================
# 5. 执行
# ==========================================
df_clean = filter_geocodable_pois(df)
df_clean.head(10)

# 2. GeoCoding

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import os
from tqdm import tqdm

# ─────────────────────────────────────────────
# 配置
# ─────────────────────────────────────────────
GEOLOC_FILE    = r"../data/processing/Geolocation_results.csv"
OUT_CSV        = r"../data/processing/regeocode_problems.csv"
CHECKPOINT_CSV = "checkpoint_geocoded_v2.csv"

AMAP_KEY      = os.environ["AMAP_API_KEY"]
MAX_RETRIES   = 3
RETRY_DELAY   = 2.0
REQUEST_DELAY = 0.5
TIMEOUT       = 15
SAVE_EVERY    = 50

WRONG_COORDS = [
    (24.48175, 118.08502),
    (24.47963, 118.08891),
]

# ─────────────────────────────────────────────
# 1. 读取原始坐标表
# ─────────────────────────────────────────────
df_xy_orign = pd.read_csv(GEOLOC_FILE, encoding='utf-8-sig')
df_xy_orign['lat'] = pd.to_numeric(df_xy_orign['lat'], errors='coerce')
df_xy_orign['lng'] = pd.to_numeric(df_xy_orign['lng'], errors='coerce')

# ─────────────────────────────────────────────
# 2. 识别问题行
# ─────────────────────────────────────────────
mask_wrong = pd.Series(False, index=df_xy_orign.index)
for lat_val, lng_val in WRONG_COORDS:
    mask_wrong |= (
        np.isclose(df_xy_orign['lat'].fillna(0), lat_val, atol=1e-4) &
        np.isclose(df_xy_orign['lng'].fillna(0), lng_val, atol=1e-4)
    )

mask_nan = df_xy_orign['lat'].isna() | df_xy_orign['lng'].isna()
mask     = mask_wrong | mask_nan

print(f"问题行 — 占位符: {mask_wrong.sum()}, NaN: {mask_nan.sum()}, 合计: {mask.sum()}")

problem_pois = df_xy_orign.loc[mask, 'location_POI'].dropna().unique()
print(f"需重新geocode的唯一POI: {len(problem_pois)}")

# ─────────────────────────────────────────────
# 3. 带重试的geocoding函数
# ─────────────────────────────────────────────
def get_amap_coords(address, city="厦门", retries=MAX_RETRIES):
    url = (f"https://restapi.amap.com/v3/geocode/geo"
           f"?address={address}&city={city}&key={AMAP_KEY}")
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, timeout=TIMEOUT).json()
            if response['status'] == '1' and int(response['count']) > 0:
                lng, lat = response['geocodes'][0]['location'].split(',')
                return float(lng), float(lat)
            else:
                return None, None
        except requests.exceptions.Timeout:
            print(f"  ⏱ 超时 (第{attempt}/{retries}次) — {address}")
            if attempt < retries:
                time.sleep(RETRY_DELAY * attempt)
        except requests.exceptions.ConnectionError:
            print(f"  🔌 连接错误 (第{attempt}/{retries}次) — {address}")
            if attempt < retries:
                time.sleep(RETRY_DELAY * attempt)
        except Exception as e:
            print(f"  ✗ 未知错误 — {address}: {e}")
            return None, None
    print(f"  ❌ 全部重试失败 — {address}")
    return None, None

# ─────────────────────────────────────────────
# 4. GCJ-02 → WGS-84 转换函数
# ─────────────────────────────────────────────
def gcj02_to_wgs84(lng, lat):
    a = 6378245.0
    ee = 0.00669342162296594323

    def transform_lat(lng, lat):
        ret = -100.0 + 2.0*lng + 3.0*lat + 0.2*lat*lat + 0.1*lng*lat + 0.2*np.sqrt(abs(lng))
        ret += (20.0*np.sin(6.0*lng*np.pi) + 20.0*np.sin(2.0*lng*np.pi)) * 2.0/3.0
        ret += (20.0*np.sin(lat*np.pi) + 40.0*np.sin(lat/3.0*np.pi)) * 2.0/3.0
        ret += (160.0*np.sin(lat/12.0*np.pi) + 320.0*np.sin(lat*np.pi/30.0)) * 2.0/3.0
        return ret

    def transform_lng(lng, lat):
        ret = 300.0 + lng + 2.0*lat + 0.1*lng*lng + 0.1*lng*lat + 0.1*np.sqrt(abs(lng))
        ret += (20.0*np.sin(6.0*lng*np.pi) + 20.0*np.sin(2.0*lng*np.pi)) * 2.0/3.0
        ret += (20.0*np.sin(lng*np.pi) + 40.0*np.sin(lng/3.0*np.pi)) * 2.0/3.0
        ret += (150.0*np.sin(lng/12.0*np.pi) + 300.0*np.sin(lng*np.pi/30.0)) * 2.0/3.0
        return ret

    dlat = transform_lat(lng - 105.0, lat - 35.0)
    dlng = transform_lng(lng - 105.0, lat - 35.0)
    radlat = lat / 180.0 * np.pi
    magic = np.sin(radlat)
    magic = 1 - ee * magic * magic
    sqrtmagic = np.sqrt(magic)
    dlat = (dlat * 180.0) / ((a * (1 - ee)) / (magic * sqrtmagic) * np.pi)
    dlng = (dlng * 180.0) / (a / sqrtmagic * np.cos(radlat) * np.pi)

    wgs_lat = lat - dlat
    wgs_lng = lng - dlng
    return wgs_lng, wgs_lat

# ─────────────────────────────────────────────
# 5. 断点续传
# ─────────────────────────────────────────────
if os.path.exists(CHECKPOINT_CSV):
    done_df    = pd.read_csv(CHECKPOINT_CSV, encoding='utf-8-sig')
    done_pois  = set(done_df['location_POI'].tolist())
    remaining  = [p for p in problem_pois if p not in done_pois]
    new_coords = done_df.to_dict('records')
    print(f"✅ 读取断点: 已完成 {len(done_pois)} 个，剩余 {len(remaining)} 个")
else:
    remaining  = list(problem_pois)
    new_coords = []
    print(f"🆕 全新开始，共 {len(remaining)} 个POI")

# ─────────────────────────────────────────────
# 6. 主循环
# ─────────────────────────────────────────────
for i, poi in enumerate(tqdm(remaining, desc="Re-geocoding")):
    lng, lat = get_amap_coords(f"厦门{poi}")
    new_coords.append({'location_POI': poi, 'new_lng': lng, 'new_lat': lat})
    time.sleep(REQUEST_DELAY)
    if (i + 1) % SAVE_EVERY == 0:
        pd.DataFrame(new_coords).to_csv(CHECKPOINT_CSV, index=False, encoding='utf-8-sig')
        print(f"  💾 检查点已保存 ({i+1}/{len(remaining)})")

pd.DataFrame(new_coords).to_csv(CHECKPOINT_CSV, index=False, encoding='utf-8-sig')
print(f"💾 最终检查点已保存 → {CHECKPOINT_CSV}")

# ─────────────────────────────────────────────
# 7. 汇总 + GCJ-02 → WGS-84 转换
# ─────────────────────────────────────────────
coords_df = pd.DataFrame(new_coords).rename(columns={'new_lng': 'lng', 'new_lat': 'lat'})
failed  = coords_df[coords_df['lng'].isna()]
success = coords_df[coords_df['lng'].notna()].copy()
print(f"\n📊 Geocode结果: 成功 {len(success)}, 失败 {len(failed)}")
if not failed.empty:
    print(f"❌ 无法geocode: {failed['location_POI'].tolist()}")

# 只对成功的行做坐标转换
wgs = success.apply(
    lambda row: pd.Series(gcj02_to_wgs84(row['lng'], row['lat']), index=['lng', 'lat']),
    axis=1
)
success[['lng', 'lat']] = wgs
print(f"✅ GCJ-02 → WGS-84 转换完成，共 {len(success)} 条")

# 合并回失败行（坐标为空）
coords_df = pd.concat([success, failed], ignore_index=True)

# ─────────────────────────────────────────────
# 8. 提取问题行完整字段，替换坐标，保存
# ─────────────────────────────────────────────
problem_rows = df_xy_orign.loc[mask].copy().drop(columns=['lng', 'lat'])
problem_rows = problem_rows.merge(
    coords_df[['location_POI', 'lng', 'lat']], on='location_POI', how='left'
)
problem_rows_valid = problem_rows.dropna(subset=['lng', 'lat'])
print(f"\n问题行: {len(problem_rows)} 条，成功修正: {len(problem_rows_valid)} 条，"
      f"仍无坐标: {len(problem_rows) - len(problem_rows_valid)} 条")

problem_rows_valid.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
print(f"✅ 已保存（WGS-84）→ {OUT_CSV}")

# 3. data filtering and merge

In [ ]:
notes = pd.read_csv(r"../data/raw/source_notes_anonymized.csv", encoding='utf-8-sig', on_bad_lines='skip', engine='python', encoding_errors='replace')
notes["笔记发布时间"].sort_values(ascending=False)
import pandas as pd
import matplotlib.pyplot as plt

# 1. Convert the column to datetime format
# If your format is unique (e.g., 2020.05), pandas usually handles it automatically
notes['笔记发布时间'] = pd.to_datetime(notes['笔记发布时间'])

# 2. Sort the data (as you requested)
notes = notes.sort_values(by='笔记发布时间', ascending=True)

# 3. Create a Year-Month period for grouping
# This makes the distribution plot cleaner by aggregating counts per month
notes['year_month'] = notes['笔记发布时间'].dt.to_period('Y').astype(str)
time_dist = notes['year_month'].value_counts().sort_index()

# 4. Plotting the distribution
plt.figure(figsize=(14, 6))
time_dist.plot(kind='line', marker='o', color='#2c3e50', linewidth=2, markersize=4)

# Formatting the chart
plt.title('Temporal Distribution of Urban Postings (May 2020 - Jan 2025)', fontsize=14)
plt.xlabel('Time (Year-Month)', fontsize=12)
plt.ylabel('Number of Postings', fontsize=12)

# Set x-ticks to show fewer labels if the timeline is too crowded
plt.xticks(range(0, len(time_dist), 3), time_dist.index[::3], rotation=45)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Save the figure
plt.savefig('data_distribution.png', dpi=300)
plt.show()

# print("The distribution plot has been generated and saved as 'data_distribution.png'.")
time_dist

In [ ]:
#  pip install wordcloud jieba matplotlib numpy pillow requests
!apt-get install -y fonts-wqy-zenhei
!pip install googletrans==4.0.0rc1
import pandas as pd
import numpy as np
import jieba
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from PIL import Image, ImageDraw
import requests
import urllib.request
from io import BytesIO
import os
from googletrans import Translator
translator = Translator()
notes = pd.read_csv(r"../data/raw/source_notes_anonymized.csv", encoding='utf-8-sig', on_bad_lines='skip', engine='python', encoding_errors='replace')

# ── 1. Extract titles ──────────────────────────────────────────────
titles = notes['笔记标题'].dropna().tolist()
text = ' '.join(titles)

# ── 2. Load HIT stopwords ─────────────────────────────────────────
HIT_STOPWORDS_URL = "https://raw.githubusercontent.com/goto456/stopwords/master/hit_stopwords.txt"

try:
    urllib.request.urlretrieve(HIT_STOPWORDS_URL, "hit_stopwords.txt")
    with open("hit_stopwords.txt", "r", encoding="utf-8") as f:
        hit_stopwords = set(line.strip() for line in f if line.strip())
    print(f"✅ Loaded {len(hit_stopwords)} HIT stopwords")
except Exception as e:
    print(f"⚠️  Could not download HIT stopwords: {e}, using fallback list")
    hit_stopwords = set(['的', '了', '在', '是', '我', '有', '和', '就', '不', '人', '都',
                         '一', '上', '也', '很', '到', '说', '要', '去', '你', '会',
                         '着', '没有', '看', '好', '这', '来', '他', '她', '们', '啊', '吧'])

extra_stopwords = {
    'citywalk', 'City', 'Walk', 'city', 'walk',
    '｜', '！', '？', '，', '??', '&', '|', '!', '?',"厦门"
}

stopwords = hit_stopwords | extra_stopwords
print(f"✅ Total stopwords: {len(stopwords)}")

# ── 3. Tokenize Chinese text with jieba ───────────────────────────
words = jieba.cut(text)
filtered = [
    w.strip() for w in words
    if w.strip()
    and w.strip() not in stopwords
    and len(w.strip()) > 1
]
word_text = ' '.join(filtered)
print(f"✅ Tokenized: {len(filtered)} words")
##  ───────────────────────── ───────────────────────── ───────────────────────── ────────────────────────
unique_words = list(set(filtered))
print(f"Translating {len(unique_words)} unique words...")

translation_map = {}
batch_size = 50  # translate in batches to avoid timeout

for i in range(0, len(unique_words), batch_size):
    batch = unique_words[i:i+batch_size]
    try:
        # Join with newline so googletrans handles as one request
        joined = '\n'.join(batch)
        result = translator.translate(joined, src='zh-cn', dest='en')
        translated = result.text.split('\n')
        for orig, trans in zip(batch, translated):
            # Clean and capitalize each translated word
            translation_map[orig] = trans.strip().replace(' ', '_')
    except Exception as e:
        print(f"⚠️  Batch {i//batch_size+1} failed: {e}, keeping originals")
        for w in batch:
            translation_map[w] = w

print("✅ Translation done")
# Replace Chinese tokens with English translations
filtered_en = [translation_map.get(w, w) for w in filtered]
word_text = ' '.join(filtered_en)

# ── 4. Create Xiamen city silhouette mask ─────────────────────────
LOCAL_MASK_PATH = '../data/processing/png/CityWalk.png'

mask_img = Image.open(LOCAL_MASK_PATH).convert('RGBA')
mask_array = np.array(mask_img)

# Use alpha channel: transparent (alpha=0) → 255 (skip), opaque (alpha>0) → 0 (fill)
alpha = mask_array[:, :, 3]
mask_wc = np.where(alpha > 0, 0, 255).astype(np.uint8)

print("✅ Mask loaded — black shape will be filled with words")

# ── 5. Load a Chinese font ────────────────────────────────────────
FONT_CANDIDATES = [
    '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc',   # Linux WQY
    '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    '/System/Library/Fonts/PingFang.ttc',              # macOS
    '/System/Library/Fonts/STHeiti Light.ttc',         # macOS fallback
    'C:/Windows/Fonts/simhei.ttf',                     # Windows SimHei
    'C:/Windows/Fonts/msyh.ttc',                       # Windows Microsoft YaHei
]

font_path = None
for fp in FONT_CANDIDATES:
    if os.path.exists(fp):
        font_path = fp
        break

if font_path is None:
    raise FileNotFoundError(
        "No Chinese font found. Install one, e.g.:\n"
        "  sudo apt install fonts-wqy-zenhei   # Ubuntu/Debian\n"
        "  brew install font-noto-cjk           # macOS via Homebrew\n"
        "or set font_path = '/your/path/to/font.ttf' manually."
    )
font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
print(f"✅ Using font: {font_path}")

# ── 6. Generate word cloud ────────────────────────────────────────
wc = WordCloud(
    font_path=font_path,
    mask=mask_wc,
    background_color='white',
    max_words=200,
    colormap='YlOrRd',
    contour_width=2,
    contour_color='#c0392b',
    width=800,
    height=600,
    prefer_horizontal=0.9,
    random_state=42,
)

wc.generate(word_text)

# ── 7. Plot & save ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 9))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('厦门笔记标题词云 · Xiamen Notes Word Cloud',
             fontsize=18, pad=16)
plt.tight_layout()
plt.savefig('xiamen_wordcloud.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Saved to xiamen_wordcloud.png")

In [ ]:
with open('../data/processing/word_cloud.txt', 'w', encoding='utf-8') as f:
    f.write(word_text)

print("✅ Saved to Google Drive")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re

# ─────────────────────────────────────────────
# 配置
# ─────────────────────────────────────────────
OUTPUT_FILE    = r"../data/processing/pad_analysis_results.csv"
GEOLOC_FILE    = r"../data/processing/Geolocation_results.csv"
GEOFIX_FILE    = r"../data/processing/regeocode_problems.csv"
CATE_FILE      = r"../data/processing/route_cate_results.csv"
EXCLUSION_FILE = r"../data/processing/broad_locations_exclusion.txt"

# ─────────────────────────────────────────────
# 读取过滤列表（兼容弯引号格式）
# ─────────────────────────────────────────────
with open(EXCLUSION_FILE, encoding='utf-8') as f:
    raw = f.read().replace('[', '').replace(']', '')

broad_locations = set(re.findall(r"['\u2018\u2019']([^'\u2018\u2019']+)['\u2018\u2019']", raw))
broad_locations.discard('')
print(f"载入过滤词: {len(broad_locations)} 条")

# ─────────────────────────────────────────────
# 辅助函数
# ─────────────────────────────────────────────
def clean_poi_data(df, column='location_POI'):
    original_count = len(df)
    df = df.dropna(subset=[column])
    df = df[~df[column].str.strip().isin(broad_locations)]
    error_pattern = r'(?i)ERROR|Invalid|Expecting value|line \d+|column \d+'
    df = df[~df[column].str.contains(error_pattern, na=False)]
    df = df[df[column].str.len() > 1]
    print(f"清理完成。原始: {original_count} 条 → 剩余: {len(df)} 条，删除: {original_count - len(df)} 条")
    return df

def clean_pad_scores(df):
    for col in ['P_score', 'A_score', 'D_score']:
        df[col] = df[col].replace({'+': 1, '-': -1}).fillna(0)
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
    return df

# ─────────────────────────────────────────────
# 1. 读取路线分类 + 饼图
# ─────────────────────────────────────────────
route_cate_df = pd.read_csv(CATE_FILE)
route_cate_df['Primary_Theme'] = route_cate_df['Primary_Theme'].replace({
    'Commercial'  : 'Commericial & Tourism',
    'Historical'  : 'Historical & Art',
    'Art'         : 'Historical & Art',
    'City_Walk'   : 'Historical & Art',
    'ERROR'       : 'Commericial & Tourism'
})

theme_counts = route_cate_df['Primary_Theme'].value_counts()
plt.figure(figsize=(8, 6))
plt.pie(theme_counts, labels=theme_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Distribution of Route Categories')
plt.axis('equal')
plt.show()

# ─────────────────────────────────────────────
# 2. 读取数据
# ─────────────────────────────────────────────
df          = pd.read_csv(OUTPUT_FILE, encoding='utf-8-sig')
df_xy_orign = pd.read_csv(GEOLOC_FILE, encoding='utf-8-sig')
df_xy_fix   = pd.read_csv(GEOFIX_FILE, encoding='utf-8-sig')

# ─────────────────────────────────────────────
# 3. 合并修正坐标 → df_xy
# ─────────────────────────────────────────────
fix = df_xy_fix[['note_index', 'location_POI', 'lng', 'lat']].rename(
    columns={'lng': 'fixed_lng', 'lat': 'fixed_lat'}
)
df_xy = df_xy_orign.merge(fix, on=['note_index', 'location_POI'], how='left')
df_xy['lng'] = df_xy['fixed_lng'].combine_first(df_xy['lng'])
df_xy['lat'] = df_xy['fixed_lat'].combine_first(df_xy['lat'])
df_xy.drop(columns=['fixed_lng', 'fixed_lat'], inplace=True)
print(f"df_xy: {df_xy.shape}")

# ─────────────────────────────────────────────
# 4. 合并PAD + 路线分类 + 坐标
# ─────────────────────────────────────────────
df = (
    df
    .merge(route_cate_df, on='note_index', how='left')
    .merge(df_xy[['note_index', 'location_POI', 'lng', 'lat']],
           on=['note_index', 'location_POI'], how='left')
)
print(f"合并后: {df.shape}")

# ─────────────────────────────────────────────
# 5. 重建location_index + 清理PAD分数
# ─────────────────────────────────────────────
df['location_index'] = df.groupby('note_index').cumcount() + 1
df = clean_pad_scores(df)

# ─────────────────────────────────────────────
# 6. 去除同一note内连续重复的POI
# ─────────────────────────────────────────────
group_id = (df['location_POI'] != df['location_POI'].shift()).cumsum()
agg_dict = {
    col: 'mean' if pd.api.types.is_numeric_dtype(df[col]) else 'first'
    for col in df.columns
}
df = df.groupby(group_id).agg(agg_dict).reset_index(drop=True)
df['location_index'] = df.groupby('note_index').cumcount() + 1
print(f"去重后: {df.shape}")

# ─────────────────────────────────────────────
# 7. 清理无效POI
# ─────────────────────────────────────────────
df = clean_poi_data(df)
wrong_coords = df.groupby(['lat','lng']).size().rename('count').sort_values(ascending=False).head(3).index
df = df[~df.set_index(['lat','lng']).index.isin(wrong_coords)].reset_index(drop=True)


print(f"最终: {df.shape}")

df.head(10)

In [ ]:
len(set(df.note_index	))

In [ ]:
df.to_csv('../data/processing/final_analysis_results_df.csv', encoding = "utf-8")

# Descriptive statistics

In [ ]:
# ----------------------------------------------------------
# Aggregation at note level
# ----------------------------------------------------------
note_agg = df.groupby('note_index').agg(
    n_locations   = ('location_POI',   'nunique'),        # unique POIs only
    dominant_theme= ('Primary_Theme',  lambda x: x.value_counts().index[0]),
    n_themes      = ('Primary_Theme',  'nunique'),
    mean_P        = ('P_score',        lambda x: pd.to_numeric(x, errors='coerce').mean()),
    mean_A        = ('A_score',        lambda x: pd.to_numeric(x, errors='coerce').mean()),
    mean_D        = ('D_score',        lambda x: pd.to_numeric(x, errors='coerce').mean()),
).reset_index()

# ============================================================
# FIGURE A — Locations per Note
# ============================================================

fig_a, ax = plt.subplots(figsize=(5, 3), facecolor='white')
ax.set_facecolor('#fafafa')

bins = range(1, 30)
ax.hist(note_agg['n_locations'], bins=bins,
        color='#4a90d9', edgecolor='white', linewidth=0.4, align='left')
ax.axvline(note_agg['n_locations'].median(), color='#e05c2a',
           linewidth=1.2, linestyle='--',
           label=f"Median = {note_agg['n_locations'].median():.0f}")
ax.axvline(note_agg['n_locations'].mean(), color='#333333',
           linewidth=1.2, linestyle=':',
           label=f"Mean = {note_agg['n_locations'].mean():.1f}")

ax.set_xlabel('Number of unique spots per note', fontsize=10)
ax.set_ylabel('Count of notes', fontsize=10)
ax.set_title('(A)  Distribution of Featured Spots per Note', fontsize=11,
             fontweight='bold', loc='left', pad=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, color='#dddddd', linewidth=0.6, linestyle='--')
ax.set_axisbelow(True)
ax.legend(fontsize=8.5, frameon=False)

plt.tight_layout()
plt.savefig('fig_A_locations.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# ============================================================
# FIGURE B — Dominant Theme per Note
# ============================================================
THEME_COLOURS = {
    'Nature'               : '#7ed81a',
    'Local_Life'           : '#ff3d00',
    'Historical & Art'     : '#ff6eb4',
    'Commericial & Tourism': '#00ccff',
}
fig_b, ax = plt.subplots(figsize=(5, 3), facecolor='white')
ax.set_facecolor('#fafafa')

theme_counts = note_agg['dominant_theme'].value_counts()
colours = [THEME_COLOURS.get(t, '#aaaaaa') for t in theme_counts.index]

bars = ax.bar(theme_counts.index, theme_counts.values,
              color=colours, edgecolor='white', linewidth=0.4, width=0.55)
for bar, val in zip(bars, theme_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            str(val), ha='center', va='bottom', fontsize=9, color='#333333')

ax.set_ylabel('Count of notes', fontsize=10)
ax.set_title('(B)  Distribution of the Primary Note Theme ', fontsize=11,
             fontweight='bold', loc='left', pad=8)
ax.set_xticklabels(theme_counts.index, rotation=15, ha='right', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, color='#dddddd', linewidth=0.6, linestyle='--')
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('fig_B_themes.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import numpy as np
import matplotlib.patches as mpatches
# ============================================================
# FIGURE D — PAD Rose Chart (Nightingale) by Dominant Theme
# ============================================================
note_agg_pad = df.groupby('note_index').agg(
    dominant_theme= ('Primary_Theme',  lambda x: x.value_counts().index[0]),
    mean_P        = ('P_score',        lambda x: pd.to_numeric(x, errors='coerce').mean()),
    mean_A        = ('A_score',        lambda x: pd.to_numeric(x, errors='coerce').mean()),
    mean_D        = ('D_score',        lambda x: pd.to_numeric(x, errors='coerce').mean()),
).reset_index()

pad_stats = note_agg_pad.groupby('dominant_theme').agg(
    mean_P = ('mean_P', 'mean'),  std_P = ('mean_P', 'std'),
    mean_A = ('mean_A', 'mean'),  std_A = ('mean_A', 'std'),
    mean_D = ('mean_D', 'mean'),  std_D = ('mean_D', 'std'),
)


pad_by_theme = note_agg.groupby('dominant_theme')[['mean_P', 'mean_A', 'mean_D']].mean()
themes  = pad_by_theme.index.tolist()
n_themes = len(themes)

# Each theme gets 3 petals (P, A, D), arranged in groups
pad_labels  = ['P', 'A', 'D']
pad_colours = ['#e05c2a', '#4a90d9', '#2ecc71']
n_petals    = n_themes * 3

angles = np.linspace(0, 2 * np.pi, n_petals, endpoint=False)
petal_width = (2 * np.pi / n_petals) * 0.75   # slight gap between petals

fig_d, ax = plt.subplots(figsize=(5, 5), subplot_kw=dict(polar=True), facecolor='white')
ax.set_facecolor('white')

# Shift zero to top
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)

# Normalise radii: shift PAD from [-1,1] to [0,1] for radius
def norm_pad(val):
    return (val + 1) / 2   # maps -1→0, 0→0.5, 1→1

pad_std_map = {'mean_P': 'std_P', 'mean_A': 'std_A', 'mean_D': 'std_D'}

for i, theme in enumerate(themes):
    for j, (pad_col, colour) in enumerate(zip(['mean_P', 'mean_A', 'mean_D'], pad_colours)):
        petal_idx = i * 3 + j
        angle     = angles[petal_idx]
        raw_val   = pad_stats.loc[theme, pad_col]
        std_val   = pad_stats.loc[theme, pad_std_map[pad_col]]
        radius    = norm_pad(raw_val) if pd.notna(raw_val) else 0
        std_r     = (std_val / 2) if pd.notna(std_val) else 0  # scale to radius space

        ax.bar(
            angle, radius,
            width=petal_width,
            bottom=0.0,
            color=colour,
            alpha=0.82,
            edgecolor='white',
            linewidth=0.5,
        )

        # --- Error bar as radial line at petal centre ---
        if std_r > 0 and radius > 0:
            ax.errorbar(
                angle, radius,
                yerr=std_r,
                fmt='none',
                ecolor='lightgrey',
                elinewidth=1.0,
                capsize=3,
                capthick=0.8,
            )

        # Value annotation
        if radius > 0.1:
            ax.text(
                angle, radius * 0.6,
                f'{raw_val:.2f}',
                ha='center', va='center',
                fontsize=7.5, color='white', fontweight='bold'
            )

# --- Theme label at centre angle of each group ---
for i, theme in enumerate(themes):
    group_centre = angles[i * 3 + 1]   # middle petal of the group
    ax.text(
        group_centre, 1.18,
        theme.replace(' & ', '\n& '),
        ha='center', va='center',
        fontsize=8.5, fontweight='bold',
        color=THEME_COLOURS.get(theme, '#333333'),
    )
# --- Divider lines between theme groups ---
for i in range(n_themes):
    # angle between last petal of group i and first petal of group i+1
    divider_angle = angles[i * 3] - petal_width / 2 - (2 * np.pi / n_petals - petal_width) / 2
    ax.plot(
        [divider_angle, divider_angle],
        [0, 1.05],
        color='#aaaaaa',
        linewidth=0.9,
        linestyle='--',
        transform=ax.transData,
        clip_on=False,
    )
# --- Radial grid labels (mapped back to PAD scale) ---
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['-0.5', '0.0', '0.5', '1.0'],
                   fontsize=7.5, color='black')
ax.set_xticks([])
ax.spines['polar'].set_visible(False)
ax.yaxis.grid(True, color='#dddddd', linewidth=0.5, linestyle='--')

# --- PAD legend ---
pad_patches = [
    mpatches.Patch(facecolor=c, edgecolor='none', label=l)
    for l, c in zip(['Pleasure (P)', 'Arousal (A)', 'Dominance (D)'], pad_colours)
]
ax.legend(handles=pad_patches, loc='lower right',
          bbox_to_anchor=(2.5, -0.08),
          frameon=False, fontsize=8.5)

ax.set_title('(C)  Distribution of Average PAD Sentiment across Notes',
             fontsize=11, fontweight='bold', color='#111111', pad=28)

plt.tight_layout()
plt.savefig('fig_D_PAD_rose.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# 4. Networking and Visualization

In [ ]:
# ============================================================
# STAGE 1 — DATA PROCESSING
# Run once; outputs self-contained GeoDataFrames for analysis
# ============================================================

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, LineString
from geopy.distance import geodesic


def build_nodes_and_links(input_df):
    """
    Process raw travel diary DataFrame into weighted node and link GeoDataFrames.

    Parameters
    ----------
    input_df : pd.DataFrame
        Raw input with columns: note_index, original_date, location_index,
        location_POI, Primary_Theme, Theme_Reasoning, lng, lat,
        P_score, A_score, D_score, reasoning, is_inherited, text_evidence

    Returns
    -------
    nodes_gdf : GeoDataFrame  (EPSG:4326)
        One row per 100-m cluster.  Columns:
            cluster_id, geometry, location_POI, Primary_Theme,
            point_weight,                   ← how many raw POI hits in cluster
            all_themes,                     ← list of all themes in cluster
            all_POIs,                       ← list of all POI names in cluster
            mean_P, mean_A, mean_D          ← average PAD scores (NaN if none)

    links_gdf : GeoDataFrame  (EPSG:4326)
        One row per unique directed cluster-pair × Theme.  Columns:
            start, end, Theme, line_weight, ← aggregated traversal count
            geometry,                       ← LineString between cluster centroids
            start_POI, end_POI,             ← representative POI names
            start_theme, end_theme          ← primary themes of endpoint clusters
    """


    # ----------------------------------------------------------
    # 1A. Clean & project
    # ----------------------------------------------------------
    df = input_df.copy()
    df['lng'] = pd.to_numeric(df['lng'], errors='coerce')
    df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
    df = df.dropna(subset=['lng', 'lat', 'note_index', 'Primary_Theme'])

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.lng, df.lat),
        crs="EPSG:4326"
    )

    # UTM 50N for metre-accurate 100 m grid clustering
    gdf_m = gdf.to_crs(epsg=32650)
    gdf_m['cluster_x'] = (gdf_m.geometry.x / 100).round() * 100
    gdf_m['cluster_y'] = (gdf_m.geometry.y / 100).round() * 100
    gdf_m['cluster_id'] = gdf_m.groupby(['cluster_x', 'cluster_y']).ngroup()

    # Propagate cluster_id back to the WGS84 frame for easy joins later
    df = df.copy()
    df['cluster_id'] = gdf_m['cluster_id'].values

    # ----------------------------------------------------------
    # 1B. Build NODES — one row per cluster, rich attributes
    # ----------------------------------------------------------
    def safe_mean(s):
        vals = pd.to_numeric(s, errors='coerce').dropna()
        return vals.mean() if len(vals) else float('nan')

    node_agg = (
        gdf_m.groupby('cluster_id')
        .apply(lambda g: pd.Series({
            'geometry'    : g.geometry.iloc[0],          # centroid proxy (first point)
            'location_POI': g['location_POI'].value_counts().index[0],
            'Primary_Theme': g['Primary_Theme'].value_counts().index[0],
            'point_weight': len(g),                       # total raw POI hits
            'all_POIs'    : g['location_POI'].tolist(),
            'all_themes'  : g['Primary_Theme'].tolist(),
            'mean_P'      : safe_mean(g['P_score']) if 'P_score' in g.columns else float('nan'),
            'mean_A'      : safe_mean(g['A_score']) if 'A_score' in g.columns else float('nan'),
            'mean_D'      : safe_mean(g['D_score']) if 'D_score' in g.columns else float('nan'),
        }), include_groups=False)
        .reset_index()
    )

    nodes_gdf = gpd.GeoDataFrame(
        node_agg, geometry='geometry', crs="EPSG:32650"
    ).to_crs(epsg=4326)

    # Convenience lookup: cluster_id → WGS84 Point
    node_coords = nodes_gdf.set_index('cluster_id')['geometry'].to_dict()

    # ----------------------------------------------------------
    # 1C. Build LINKS — directed edges from sequential itinerary
    # ----------------------------------------------------------
    raw_links = []
    sorted_df = df.sort_values(['note_index', 'location_index'])

    for note_id, group in sorted_df.groupby('note_index'):
        theme = group['Primary_Theme'].iloc[0]
        cluster_seq = group['cluster_id'].tolist()

        for i in range(len(cluster_seq) - 1):
            s_id, e_id = cluster_seq[i], cluster_seq[i + 1]
            if s_id == e_id:
                continue
            p1, p2 = node_coords[s_id], node_coords[e_id]
            dist_m = geodesic((p1.y, p1.x), (p2.y, p2.x)).meters
            if dist_m < 1500:                             # jump guard: 1.5 km
                raw_links.append({
                    'start'     : s_id,
                    'end'       : e_id,
                    'Theme'     : theme,
                    'note_index': note_id,
                })

    links_df = pd.DataFrame(raw_links)

    if not links_df.empty:
        agg_links = (
              links_df
              .groupby(['start', 'end'])
              .agg(
                  line_weight=('note_index', 'count'),
                  note_ids   =('note_index', list),
                  Theme      =('Theme', lambda x: x.value_counts().index[0]),  # 主导theme
              )
              .reset_index()
          )

        # Attach geometry and endpoint metadata
        node_meta = nodes_gdf.set_index('cluster_id')[['location_POI', 'Primary_Theme']]

        agg_links['geometry']    = agg_links.apply(
            lambda r: LineString([node_coords[r['start']], node_coords[r['end']]]), axis=1
        )
        agg_links['start_POI']   = agg_links['start'].map(node_meta['location_POI'])
        agg_links['end_POI']     = agg_links['end'].map(node_meta['location_POI'])
        agg_links['start_theme'] = agg_links['start'].map(node_meta['Primary_Theme'])
        agg_links['end_theme']   = agg_links['end'].map(node_meta['Primary_Theme'])

        links_gdf = gpd.GeoDataFrame(agg_links, geometry='geometry', crs="EPSG:4326")
    else:
        links_gdf = gpd.GeoDataFrame(
            columns=['start', 'end', 'Theme', 'line_weight', 'note_ids',
                     'geometry', 'start_POI', 'end_POI', 'start_theme', 'end_theme'],
            crs="EPSG:4326"
        )
    print("sample start values:", agg_links['start'].head().tolist())
    print("node_coords keys sample:", list(node_coords.keys())[:5])
    print("overlap:", len(set(agg_links['start']) & set(node_coords.keys())))
    print("links start not in nodes:", set(agg_links['start']) - set(node_coords.keys()))
    return nodes_gdf, links_gdf


# --- RUN ONCE ---
final_nodes_df, final_lines_df = build_nodes_and_links(df)


print("Nodes:", len(final_nodes_df))
print(final_nodes_df.columns.tolist())
print("\nLinks:", len(final_lines_df))
print(final_lines_df.columns.tolist())

In [ ]:
final_nodes_df.to_file(r"../data/processing/final_nodes_df", encoding='utf-8-sig')
final_lines_df.to_file(r"../data/processing/final_lines_df", encoding='utf-8-sig')

In [ ]:
from folium import plugins
import folium
import geopandas as gpd

# ----------------------------------------------------------
# 1. Load district boundaries from local shapefile
# ----------------------------------------------------------
districts_raw = gpd.read_file(
    '../data/processing/xiamen_district'
)
print(districts_raw.columns.tolist())
print(districts_raw[['NAME', 'geometry']].head())

# project to UTM 50N for dissolve
xiamen_districts = districts_raw.to_crs(epsg=32650)

# dissolve all districts → city boundary
xiamen_city = xiamen_districts.dissolve().reset_index()

# convert back to WGS84 for folium
xiamen_districts_wgs = xiamen_districts.to_crs(epsg=4326)
xiamen_city_wgs      = xiamen_city.to_crs(epsg=4326)

print(f"\nDistricts : {len(xiamen_districts_wgs)}")
print(f"City boundary : {len(xiamen_city_wgs)}")


# ----------------------------------------------------------
# 2. Visualisation function
# ----------------------------------------------------------
def visualize_network(nodes_gdf, links_gdf,
                      city_gdf=None, districts_gdf=None):

    theme_colour = {
        'Nature'               : '#7ed81a',
        'Local_Life'           : '#ff3d00',
        'Historical & Art'     : '#ff3fa4',
        'Commericial & Tourism': '#00ccff',
    }

    # --- Initialise map ---
    m = folium.Map(
        location=[nodes_gdf.geometry.y.mean(), nodes_gdf.geometry.x.mean()],
        zoom_start=14,
        tiles='CartoDB dark_matter'
    )

    # --- Scale bar ---
    plugins.MeasureControl(
        position='bottomleft',
        primary_length_unit='meters',
        secondary_length_unit='kilometers',
    ).add_to(m)

    # --- North arrow ---
    north_arrow_html = """
    <div style="
        position:fixed; bottom:120px; left:20px; z-index:1000;
        background:rgba(30,30,30,0.75); border-radius:6px;
        padding:6px 8px; text-align:center; font-family:Arial;">
        <div style="font-size:22px; color:white; line-height:1;">&#8679;</div>
        <div style="font-size:10px; color:#cccccc; letter-spacing:1px;
             margin-top:2px;">N</div>
    </div>"""
    m.get_root().html.add_child(folium.Element(north_arrow_html))

    # --- City boundary ---
    if city_gdf is not None:
        folium.GeoJson(
            city_gdf.__geo_interface__,
            name='Xiamen City Boundary',
            style_function=lambda _: {
                'color'      : '#ffffff',
                'weight'     : 2.5,
                'fillColor'  : 'none',
                'fillOpacity': 0,
                'dashArray'  : '6 4',
            },
            tooltip='Xiamen City',
        ).add_to(m)

    # --- District boundaries + NAME labels ---
    if districts_gdf is not None:
        for _, district in districts_gdf.iterrows():
            folium.GeoJson(
                district.geometry.__geo_interface__,
                name='District',
                style_function=lambda _: {
                    'color'      : '#aaaaaa',
                    'weight'     : 1.2,
                    'fillColor'  : 'none',
                    'fillOpacity': 0,
                    'dashArray'  : '3 3',
                },
            ).add_to(m)

            centroid = district.geometry.centroid
            folium.Marker(
                location=[centroid.y, centroid.x],
                icon=folium.DivIcon(
                    html=f"""
                    <div style="
                        font-size:11px;
                        font-family:Arial;
                        color:#cccccc;
                        font-weight:bold;
                        letter-spacing:1px;
                        text-shadow: 1px 1px 2px #000000;
                        white-space:nowrap;
                    ">{district['NAME']}</div>
                    """,
                    icon_size=(120, 20),
                    icon_anchor=(60, 10),
                ),
            ).add_to(m)

    # --- Legend ---
    legend_items = ''.join([
        f"""<div style="display:flex;align-items:center;margin-bottom:6px;">
            <div style="width:12px;height:12px;border-radius:50%;
                background:{colour};margin-right:8px;flex-shrink:0;"></div>
            <span style="font-size:11px;color:#eeeeee;">{theme}</span>
        </div>"""
        for theme, colour in theme_colour.items()
    ])

    legend_html = f"""
    <div style="
        position:fixed; bottom:30px; right:20px; z-index:1000;
        background:rgba(30,30,30,0.85); border-radius:8px;
        padding:12px 14px; font-family:Arial; min-width:190px;
        box-shadow:0 2px 8px rgba(0,0,0,0.4);">
        <div style="font-size:12px;font-weight:bold;color:white;
            margin-bottom:10px;border-bottom:1px solid #555;
            padding-bottom:6px;">Map Legend</div>
        {legend_items}
        <div style="margin-top:10px;border-top:1px solid #555;padding-top:8px;">
            <div style="display:flex;align-items:center;margin-bottom:4px;">
                <div style="width:30px;height:0px;border-top:2px dashed #ffffff;
                    margin-right:8px;"></div>
                <span style="font-size:10px;color:#aaaaaa;">City boundary</span>
            </div>
            <div style="display:flex;align-items:center;margin-bottom:8px;">
                <div style="width:30px;height:0px;border-top:1px dashed #aaaaaa;
                    margin-right:8px;"></div>
                <span style="font-size:10px;color:#aaaaaa;">District boundary</span>
            </div>
            <div style="font-size:10px;color:#aaaaaa;margin-bottom:2px;">
                Node size = spots frequency</div>
            <div style="font-size:10px;color:#aaaaaa;">
                Line width = path frequency</div>
        </div>
    </div>"""
    m.get_root().html.add_child(folium.Element(legend_html))

    # --- Lines ---
    theme_order = ['Nature', 'Commericial & Tourism', 'Local_Life', 'Historical & Art']
    links_sorted = links_gdf.copy()
    links_sorted['_order'] = links_sorted['Theme'].map(
        {t: i for i, t in enumerate(theme_order)}
    ).fillna(0)
    links_sorted = links_sorted.sort_values('_order').drop(columns='_order')

    for _, row in links_sorted.iterrows():
        line_pts = [[p[1], p[0]] for p in list(row['geometry'].coords)]
        clamped_weight = max(1, min(row['line_weight'] * 1.5, 8))
        folium.PolyLine(
            locations=line_pts,
            color=theme_colour.get(row['Theme'], 'white'),
            weight=clamped_weight,
            opacity=0.5,
            tooltip=f"Theme: {row['Theme']}<br>Path Frequency: {row['line_weight']}"
        ).add_to(m)

    # --- Nodes ---
    nodes_sorted = nodes_gdf.copy()
    nodes_sorted['_order'] = nodes_sorted['Primary_Theme'].map(
        {t: i for i, t in enumerate(theme_order)}
    ).fillna(0)
    nodes_sorted = nodes_sorted.sort_values('_order').drop(columns='_order')

    for _, row in nodes_sorted.iterrows():
        node_radius = max(1, min(1 + (row['point_weight'] * 0.1), 6))
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=node_radius,
            color='white', weight=0.5,
            fill=True,
            fill_color=theme_colour.get(row['Primary_Theme'], 'gray'),
            fill_opacity=0.75,
            popup=f"<b>POI:</b> {row['location_POI']}<br>"
                  f"<b>Spots frequency:</b> {row['point_weight']}"
        ).add_to(m)

    return m


# ----------------------------------------------------------
# 3. Run
# ----------------------------------------------------------
final_nodes_df = final_nodes_df.sort_values('point_weight', ascending=False)
final_nodes_df = final_nodes_df.drop(final_nodes_df.index[[0, 2]]).reset_index(drop=True)

result_map = visualize_network(
    final_nodes_df[final_nodes_df['point_weight'] > 1],
    final_lines_df,
    city_gdf=xiamen_city_wgs,
    districts_gdf=xiamen_districts_wgs,
)

result_map.save("result_map.html")
result_map


# Results Analysis

## Top10 Analysis

In [ ]:
# final_nodes_df.to_csv(r"../data/processing/final_nodes_df.csv", encoding='utf-8-sig')
print(final_nodes_df.sort_values('point_weight', ascending=False).iloc[0]['all_POIs'])
final_nodes_df.sort_values('point_weight', ascending=False).head(10)

In [ ]:
final_nodes_df.sort_values('point_weight', ascending=False).index.get_loc(
    final_nodes_df[final_nodes_df['location_POI'] == '陈嘉庚故居'].index[0]
) + 1

In [ ]:
final_nodes_df[final_nodes_df['Primary_Theme'] == 'Local_Life'].sort_values('point_weight', ascending=False).head(10)

In [ ]:
top10

### NodeLevel

In [ ]:
# ============================================================
# FIGURE 1 — Bar chart: Top 10 POIs by Visit Frequency
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

THEME_COLOURS = {
    'Nature'               : '#7ed81a',
    'Local_Life'           : '#ff3d00',
    'Historical & Art'     : '#ff6eb4',
    'Commericial & Tourism': '#00ccff',
}
DEFAULT_COLOUR = '#aaaaaa'

POI_TRANSLATION = {
    '十里长堤'     : 'Shili Long Causeway',
    '溪岸路花鸟市场': 'Xianan Road Flower & Bird Market',
    '中山路'       : 'Zhongshan Road',
    '黄厝沙滩'     : 'Island Ring Boulevard',
    '厦门大学'     : 'Xiamen University',
    '沙坡尾'       : 'Shapowei Art Village',
    '幸福路'       : 'Xingfu Road',
    '鼓浪屿'       : 'Gulangyu Island',
    '华新路'       : 'Huaxin Road',
    '中山公园'     : 'Zhongshan Park',
    '八市'         : 'The Eighth Market',
    '陈嘉庚故居'   : 'Former Residence of Tan Kah Kee',
    '百家村'       : 'Baijia Village',
    '南普陀寺'     : 'Nanputuo Temple',
    '集美学村'     : 'Jimei School Village',
}

plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 10,
    'axes.linewidth'   : 0.8,
    'axes.edgecolor'   : '#333333',
    'xtick.direction'  : 'out',
    'ytick.direction'  : 'out',
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
})

def translate_poi(name):
    return POI_TRANSLATION.get(name, name)

# ============================================================
# FIGURE 1 — Bar chart: Top 10 POIs by Visit Frequency
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

THEME_COLOURS = {
    'Nature'               : '#7ed81a',
    'Local_Life'           : '#ff3d00',
    'Historical & Art'     : '#ff6eb4',
    'Commericial & Tourism': '#00ccff',
}
DEFAULT_COLOUR = '#aaaaaa'

POI_TRANSLATION = {
    '十里长堤'     : 'Shili Long Causeway',
    '溪岸路花鸟市场': "Xi'an Road Flower &\n Bird Market",
    '中山路'       : 'Zhongshan Road',
    '黄厝沙滩'     : 'Island Ring Boulevard',
    '厦门大学'     : 'Xiamen University',
    '沙坡尾'       : 'Shapowei Art Village',
    '幸福路'       : 'Xingfu Road',
    '鼓浪屿'       : 'Gulangyu Island',
    '华新路'       : 'Huaxin Road',
    '中山公园'     : 'Zhongshan Park',
    '八市'         : 'The Eighth Market',
    '陈嘉庚故居'   : 'Former Residence of Tan Kah Kee',
    '百家村'       : 'Baijia Village',
    '南普陀寺'     : 'Nanputuo Temple',
    '集美学村'     : 'Jimei School Village',
}

plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 10,
    'axes.linewidth'   : 0.8,
    'axes.edgecolor'   : '#333333',
    'xtick.direction'  : 'out',
    'ytick.direction'  : 'out',
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
})

def translate_poi(name):
    return POI_TRANSLATION.get(name, name)

# ----------------------------------------------------------
# Data
# ----------------------------------------------------------
top8 = (
    final_nodes_df
    .sort_values('point_weight', ascending=False)
    .head(8)
    .reset_index(drop=True)
    .copy()
)
top8['_is_extra'] = False

extra_theme_order = ['Local_Life', 'Historical & Art']
extra_rows = []
for theme in extra_theme_order:
    if theme not in set(top8['Primary_Theme']):
        top_of_theme = (
            final_nodes_df[
                (final_nodes_df['Primary_Theme'] == theme) &
                (~final_nodes_df.index.isin(top8.index))
            ]
            .sort_values('point_weight', ascending=False)
            .iloc[0]
        )
        extra_rows.append(top_of_theme)

extra_df = pd.DataFrame(extra_rows).reset_index(drop=True)
extra_df['_is_extra'] = True

top10 = pd.concat([top8, extra_df], ignore_index=True)
top10['poi_en'] = top10['location_POI'].apply(translate_poi)
top10['theme_counts'] = top10['all_themes'].apply(
    lambda lst: pd.Series(lst).value_counts().to_dict()
)
plot_df = top10

# ----------------------------------------------------------
# Plot variables
# ----------------------------------------------------------
pois     = plot_df['poi_en'].tolist()
weights  = plot_df['point_weight'].tolist()
bcolors  = [THEME_COLOURS.get(t, DEFAULT_COLOUR) for t in plot_df['Primary_Theme']]
is_extra = plot_df['_is_extra'].tolist()
n_extra  = sum(is_extra)
max_w    = max(weights)

# ----------------------------------------------------------
# Figure — single axis, no broken axis
# ----------------------------------------------------------
fig1, ax = plt.subplots(figsize=(10, 7), facecolor='white')
ax.set_facecolor('#fafafa')

bars = ax.barh(pois[::-1], weights[::-1], color=bcolors[::-1],
               height=0.55, edgecolor='white', linewidth=0.4)

# --- Value labels + hatch ---
for bar, val, extra in zip(bars, weights[::-1], is_extra[::-1]):
    label = str(val) + (' †' if extra else '')
    ax.text(
        bar.get_width() + max_w * 0.01,
        bar.get_y() + bar.get_height() / 2,
        label, va='center', ha='left', color='#333333', fontsize=9,
    )
    if extra:
        bar.set_hatch('///')
        bar.set_edgecolor('#888888')
        bar.set_linewidth(0.5)

# --- Separator line + annotation ---
if n_extra > 0:
    sep_y = n_extra - 0.5
    ax.axhline(y=sep_y, color='#aaaaaa', linewidth=0.9, linestyle='--')
    ax.text(
        max_w * 0.8, sep_y + 0.15,
        '† Included for thematic diversity',
        va='bottom', ha='left',
        color='#888888', fontsize=9, style='italic'
    )

# --- Axes styling ---
ax.set_xlim(0, max_w * 1.15)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.set_xlabel('Number of featured notes', color='#333333',
              fontsize=10, labelpad=6)
ax.set_title('(A)  Top Featured Spots in Xiamen',
             color='#111111', fontsize=12, fontweight='bold',
             loc='left', pad=10)
ax.tick_params(axis='both', colors='#333333', labelsize=9)
ax.tick_params(axis='y', length=0)
ax.xaxis.grid(True, color='#dddddd', linewidth=0.6, linestyle='--')
ax.set_axisbelow(True)

# --- Legend ---
legend_patches = [
    mpatches.Patch(facecolor=c, edgecolor='#999999', linewidth=0.5, label=t)
    for t, c in THEME_COLOURS.items()
]
legend = ax.legend(
    handles=legend_patches, loc='lower right',
    frameon=True, framealpha=0.95, edgecolor='#cccccc',
    fontsize=8.5, title='Primary Theme', title_fontsize=8.5,
)
legend.get_title().set_horizontalalignment('center')

plt.tight_layout()
plt.savefig('fig1_bar.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# ============================================================
# FIGURE 2 — Pie grid: Theme distribution per POI (2 rows × 5 cols)
# ============================================================

fig2, axes = plt.subplots(
    nrows=2, ncols=5,
    figsize=(9, 4),
    facecolor='white',
)
fig2.patch.set_facecolor('white')

for idx, row in top10.iterrows():
    r = idx // 5
    c = idx % 5
    ax = axes[r, c]
    ax.set_facecolor('white')

    tc      = row['theme_counts']
    labels  = list(tc.keys())
    sizes   = list(tc.values())
    colours = [THEME_COLOURS.get(l, DEFAULT_COLOUR) for l in labels]

    _, _, autotexts = ax.pie(
        sizes,
        labels=None,
        colors=colours,
        autopct=lambda p: f'{p:.0f}%' if p > 6 else '',
        startangle=90,
        wedgeprops=dict(linewidth=0.5, edgecolor='white'),
        pctdistance=0.72,
        radius=0.85,
    )
    for at in autotexts:
        at.set_color('#111111')
        at.set_fontsize(8)

    primary = row['Primary_Theme']
    ax.set_title(
        f'({range(1,11)[idx]}) {row["poi_en"]}',
        color='#111111', fontsize=8.5,
        loc='left', pad=0, linespacing=1,
    )
fig2.suptitle(
    '(B) Thematic Composition of Representative Spots',
    color='#111111', fontsize=8.5, fontweight='bold',
    x=0, ha='left', y=1.01,
)
plt.tight_layout(h_pad=1, w_pad=1)
plt.savefig('fig2_pies.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# ============================================================
# FIGURE 3 — Diverging Heatmap: PAD Emotional Profile
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 10,
    'axes.linewidth'   : 0.8,
})

# --- Data ---
pad_df = top10[['poi_en', 'Primary_Theme', 'mean_P', 'mean_A', 'mean_D']].copy()
pad_df.columns = ['POI', 'Theme', 'Pleasure', 'Arousal', 'Dominance']
pad_df = pad_df.set_index('POI')

pad_values = pad_df[['Pleasure', 'Arousal', 'Dominance']].astype(float)

# --- Figure ---
fig3, ax = plt.subplots(figsize=(7, 6), facecolor='white')
ax.set_facecolor('white')

# Diverging colormap: negative=blue, zero=white, positive=red
cmap = plt.cm.RdBu_r
norm = mcolors.TwoSlopeNorm(vmin=-1.0, vcenter=0.0, vmax=1.0)

im = ax.imshow(
    pad_values.values,
    cmap=cmap,
    norm=norm,
    aspect='auto',
)

# --- Axis labels ---
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Pleasure\n(P)', 'Arousal\n(A)', 'Dominance\n(D)'],
                   fontsize=10, fontweight='bold', color='#111111')

ax.set_yticks(range(len(pad_df)))
ax.set_yticklabels(
    [f'{poi}' for poi in pad_df.index],
    fontsize=9, color='#111111'
)

# --- Colour y-tick labels by primary theme ---
for tick, poi in zip(ax.get_yticklabels(), pad_df.index):
    theme = pad_df.loc[poi, 'Theme']
    tick.set_color(THEME_COLOURS.get(theme, '#333333'))
    tick.set_fontweight('bold')

# --- Annotate each cell with value ---
for i in range(len(pad_df)):
    for j in range(3):
        val = pad_values.iloc[i, j]
        text_color = 'white' if abs(val) > 0.55 else '#111111'
        ax.text(
            j, i, f'{val:.2f}',
            ha='center', va='center',
            fontsize=8.5, color=text_color,
        )

# --- Grid lines between cells ---
ax.set_xticks(np.arange(-0.5, 3, 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(pad_df), 1), minor=True)
ax.grid(which='minor', color='white', linewidth=1.5)
ax.tick_params(which='minor', length=0)
ax.tick_params(which='major', length=0)

# --- Colorbar ---
cbar = fig3.colorbar(im, ax=ax, orientation='vertical',
                     fraction=0.03, pad=0.02)
cbar.set_label('PAD Score', fontsize=9, color='#333333')
cbar.ax.tick_params(labelsize=8, colors='#333333')
cbar.outline.set_visible(False)

# --- Theme colour legend for y-axis ---
legend_patches = [
    mpatches.Patch(facecolor=c, edgecolor='none', label=t)
    for t, c in THEME_COLOURS.items()
]

ax.spines[:].set_visible(False)

ax.set_title('(C)  PAD Sentiment Profile across Representative Spots',
             fontsize=11, fontweight='bold', color='#111111',
             loc='left', pad=10)

plt.tight_layout()
plt.savefig('fig3_pad_heatmap.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

### RouteLevel

In [ ]:
final_lines_df.sort_values('line_weight', ascending=False).head(10)

In [ ]:
# ============================================================
# LINES ANALYSIS — Top 10 most frequent paths
# ============================================================

# ----------------------------------------------------------
# Data — top 8 + 2 theme representatives
# ----------------------------------------------------------
top8_lines = (
    final_lines_df
    .sort_values('line_weight', ascending=False)
    .head(8)
    .reset_index(drop=True)
    .copy()
)
top8_lines['_is_extra'] = False

extra_theme_order_lines = ['Historical & Art','Local_Life']
extra_rows_lines = []
for theme in extra_theme_order_lines:
    if theme not in set(top8_lines['Theme']):
        top_of_theme = (
            final_lines_df[
                (final_lines_df['Theme'] == theme) &
                (~final_lines_df.index.isin(top8_lines.index))
            ]
            .sort_values('line_weight', ascending=False)
            .iloc[0]
        )
        extra_rows_lines.append(top_of_theme)

if extra_rows_lines:
    extra_lines_df = pd.DataFrame(extra_rows_lines).reset_index(drop=True)
    extra_lines_df['_is_extra'] = True
    top10_lines = pd.concat([top8_lines, extra_lines_df], ignore_index=True)
else:
    top10_lines = top8_lines.copy()

# Build readable edge label: "StartPOI → EndPOI"
top10_lines['edge_en'] = top10_lines.apply(
    lambda r: f"{translate_poi(r['start_POI'])} →\n{translate_poi(r['end_POI'])}",
    axis=1
)

# Theme distribution along each edge — from note_ids
def edge_theme_counts(note_ids_list):
    themes = []
    for nid in note_ids_list:
        match = df[df['note_index'] == nid]['Primary_Theme']
        if len(match):
            themes.append(match.value_counts().index[0])
    return pd.Series(themes).value_counts().to_dict()

top10_lines['theme_counts'] = top10_lines['note_ids'].apply(edge_theme_counts)

In [ ]:
# ==============================================================================
# 1. 构建标准 POI 译名库 (POI_TRANSLATION_lines)
# 原则：官方优先 > 通用旅游译名 > 汉语拼音 (专名 + 通名)
# ==============================================================================
POI_TRANSLATION_lines = {
    # === 自然与景区类 (Nature & Attractions) ===
    "植物园": "Xiamen Botanical Garden",      # 加城市限定词，避免全球歧义
    "日光岩": "Sunlight Rock",                # 鼓浪屿地标标准译名 (官方)
    "菽庄花园": "Shuzhuang Garden",           # 专名拼音 + 通名英文
    "南普陀寺": "Nanputuo Temple",            # 宗教场所标准译法
    "钟鼓索道": "Zhonggu Cable Car",          # 交通设施：拼音专名 + 功能通名
    "十里长堤": "Shili Causeway",             # 地理术语：Causeway 已含"长堤"义
    "龙舟池": "Dragon Boat Pool",             # 功能性地标直译
    # === 教育与文化类 (Education & Culture) ===
    "厦门大学": "Xiamen University",          # 官方英文校名
    "集美学村": "Jimei School Village",       # 历史街区固定译名
    "嘉庚公园": "Jiageng Park",               # 人名地名：拼音 + Park
    # === 城市公共空间 (Public Spaces) ===
    "中山公园": "Zhongshan Park",             # 标准公园译法
    "音乐广场": "Music Square",               # 标准广场译法
    "溪头下": "Xitouxia",                     # 村落/片区名：纯拼音保留原真性
    # === 道路与街巷 (Roads & Streets) ===
    "华新路": "Huaxin Road",                  # 拼音专名 + Road (DB3502 规范)
    "幸福路": "Xingfu Road",                  # 同上
    "公园西路": "Gongyuan West Road",         # 方位词标准译法：West + Road
    "湖明路":"Huming Road"
}

# ==============================================================================
# 2. 定义翻译与更新函数
# ==============================================================================
def get_standard_name(cn_name, translation_dict):
    """
    查找标准英文名，若未找到则返回原中文（作为 fallback）
    """
    return translation_dict.get(cn_name, cn_name)

def update_edge_translations(df, trans_dict):
    """
    根据 start_POI 和 end_POI 重新生成标准化的 edge_en 列
    格式：Start Name →\nEnd Name
    """
    def format_edge(row):
        start_en = get_standard_name(row['start_POI'], trans_dict)
        end_en = get_standard_name(row['end_POI'], trans_dict)
        return f"{start_en} →\n{end_en}"

    # 创建新列或覆盖旧列
    df['edge_en'] = df.apply(format_edge, axis=1)
    return df

# 假设 top10_lines 是您的 DataFrame 变量名
top10_lines = update_edge_translations(top10_lines, POI_TRANSLATION_lines)

# 打印前几行检查结果
top10_lines

In [ ]:
# ============================================================
# FIGURE 4 — Bar chart: Top 10 Paths by Traversal Frequency
# ============================================================

plot_lines_df = top10_lines
pois_l    = plot_lines_df['edge_en'].tolist()
weights_l = plot_lines_df['line_weight'].tolist()
bcolors_l = [THEME_COLOURS.get(t, DEFAULT_COLOUR) for t in plot_lines_df['Theme']]
is_extra_l= plot_lines_df['_is_extra'].tolist()
n_extra_l = sum(is_extra_l)
max_wl    = max(weights_l)

fig4, ax = plt.subplots(figsize=(11, 7), facecolor='white')
ax.set_facecolor('#fafafa')

bars = ax.barh(pois_l[::-1], weights_l[::-1], color=bcolors_l[::-1],
               height=0.55, edgecolor='white', linewidth=0.4)

for bar, val, extra in zip(bars, weights_l[::-1], is_extra_l[::-1]):
    label = str(val) + (' †' if extra else '')
    ax.text(
        bar.get_width() + max_wl * 0.01,
        bar.get_y() + bar.get_height() / 2,
        label, va='center', ha='left', color='#333333', fontsize=9,
    )
    if extra:
        bar.set_hatch('///')
        bar.set_edgecolor('#888888')
        bar.set_linewidth(0.5)

if n_extra_l > 0:
    sep_y = n_extra_l - 0.5
    ax.axhline(y=sep_y, color='#aaaaaa', linewidth=0.9, linestyle='--')
    ax.text(
        max_wl * 0.5, sep_y + 0.15,
        '† Included for thematic diversity',
        va='bottom', ha='left',
        color='#888888', fontsize=9, style='italic'
    )

ax.set_xlim(0, max_wl * 1.15)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.set_xlabel('Number of featured notes', color='#333333',
              fontsize=10, labelpad=6)
ax.set_title('(D)  Top Featured Routes in Xiamen',
             color='#111111', fontsize=12, fontweight='bold',
             loc='left', pad=10)
ax.tick_params(axis='both', colors='#333333', labelsize=9)
ax.tick_params(axis='y', length=0)
ax.xaxis.grid(True, color='#dddddd', linewidth=0.6, linestyle='--')
ax.set_axisbelow(True)

legend_patches = [
    mpatches.Patch(facecolor=c, edgecolor='#999999', linewidth=0.5, label=t)
    for t, c in THEME_COLOURS.items()
]
legend = ax.legend(
    handles=legend_patches, loc='lower right',
    frameon=True, framealpha=0.95, edgecolor='#cccccc',
    fontsize=8.5, title='Theme', title_fontsize=8.5,
)
legend.get_title().set_horizontalalignment('center')

plt.tight_layout()
plt.savefig('fig4_lines_bar.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# ============================================================
# FIGURE 5 — Pie grid: Theme distribution per path (2×5)
# ============================================================

fig5, axes = plt.subplots(nrows=2, ncols=5, figsize=(9, 4), facecolor='white')
fig5.patch.set_facecolor('white')

for idx, row in top10_lines.iterrows():
    r = idx // 5
    c = idx % 5
    ax = axes[r, c]
    ax.set_facecolor('white')

    tc      = row['theme_counts']
    labels  = list(tc.keys())
    sizes   = list(tc.values())
    colours = [THEME_COLOURS.get(l, DEFAULT_COLOUR) for l in labels]

    _, _, autotexts = ax.pie(
        sizes, labels=None, colors=colours,
        autopct=lambda p: f'{p:.0f}%' if p > 6 else '',
        startangle=90,
        wedgeprops=dict(linewidth=0.5, edgecolor='white'),
        pctdistance=0.72, radius=0.85,
    )
    for at in autotexts:
        at.set_color('#111111')
        at.set_fontsize(8)

    ax.set_title(
        f'({idx + 1})  {translate_poi(row["edge_en"])}',
        color='#111111', fontsize=7.5, fontweight='bold',
        loc='center', pad=0, linespacing=1.3,
    )

plt.tight_layout(h_pad=1, w_pad=1)
plt.savefig('fig5_lines_pies.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# ============================================================
# FIGURE 6 — Diverging Heatmap: PAD profile per top path
# ============================================================

# PAD per edge: average across all notes that traverse it
def edge_pad_mean(note_ids_list, dim):
    vals = []
    for nid in note_ids_list:
        match = pd.to_numeric(
            df[df['note_index'] == nid][dim], errors='coerce'
        ).dropna()
        if len(match):
            vals.append(match.mean())
    return np.mean(vals) if vals else float('nan')

for dim in ['P_score', 'A_score', 'D_score']:
    top10_lines[f'mean_{dim[0]}'] = top10_lines['note_ids'].apply(
        lambda ids: edge_pad_mean(ids, dim)
    )

pad_lines = top10_lines[['edge_en', 'Theme',
                          'mean_P', 'mean_A', 'mean_D']].copy()
pad_lines.columns = ['Edge', 'Theme', 'Pleasure', 'Arousal', 'Dominance']
pad_lines = pad_lines.set_index('Edge')
pad_vals  = pad_lines[['Pleasure', 'Arousal', 'Dominance']].astype(float)

fig6, ax = plt.subplots(figsize=(7, 6), facecolor='white')
ax.set_facecolor('white')

cmap = plt.cm.RdBu_r
norm = mcolors.TwoSlopeNorm(vmin=-1.0, vcenter=0.0, vmax=1.0)
im = ax.imshow(pad_vals.values, cmap=cmap, norm=norm, aspect='auto')

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Pleasure\n(P)', 'Arousal\n(A)', 'Dominance\n(D)'],
                   fontsize=10, fontweight='bold', color='#111111')
ax.set_yticks(range(len(pad_lines)))
ax.set_yticklabels(pad_lines.index, fontsize=8, color='#111111')

for tick, edge in zip(ax.get_yticklabels(), pad_lines.index):
    theme = pad_lines.loc[edge, 'Theme']
    tick.set_color(THEME_COLOURS.get(theme, '#333333'))
    tick.set_fontweight('bold')

for i in range(len(pad_lines)):
    for j in range(3):
        val = pad_vals.iloc[i, j]
        if pd.notna(val):
            text_color = 'white' if abs(val) > 0.55 else '#111111'
            ax.text(j, i, f'{val:.2f}',
                    ha='center', va='center',
                    fontsize=8.5, color=text_color)

ax.set_xticks(np.arange(-0.5, 3, 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(pad_lines), 1), minor=True)
ax.grid(which='minor', color='white', linewidth=1.5)
ax.tick_params(which='minor', length=0)
ax.tick_params(which='major', length=0)

cbar = fig6.colorbar(im, ax=ax, orientation='vertical',
                     fraction=0.03, pad=0.02)
cbar.set_label('PAD Score', fontsize=9, color='#333333')
cbar.ax.tick_params(labelsize=8, colors='#333333')
cbar.outline.set_visible(False)

ax.spines[:].set_visible(False)
ax.set_title('(F)  PAD Emotional Profile across Routes',
             fontsize=11, fontweight='bold', color='#111111',
             loc='left', pad=10)

plt.tight_layout()
plt.savefig('fig6_lines_pad.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## Geo-Network analysis

In [ ]:
# ============================================================
# RANK-SIZE ANALYSIS — Local network parameters
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import networkx as nx

# ----------------------------------------------------------
# 1. Compute weighted centrality metrics
# ----------------------------------------------------------
degree_w      = dict(G.degree(weight='weight'))
closeness_w   = nx.closeness_centrality(G, distance='weight')
betweenness_w = nx.betweenness_centrality(G, weight='weight', normalized=True)
eigenvector_w = nx.eigenvector_centrality(G, weight='weight', max_iter=1000)

pagerank_w = nx.pagerank(G, weight='weight')

centrality_df = pd.DataFrame({
    'cluster_id'  : list(G.nodes()),
    'degree'      : [degree_w[n]     for n in G.nodes()],
    'closeness'   : [closeness_w[n]  for n in G.nodes()],
    'betweenness' : [betweenness_w[n] for n in G.nodes()],
    'pagerank'    : [pagerank_w[n]   for n in G.nodes()],
}).query('degree > 0 and closeness > 0 and '
         'betweenness > 0 and pagerank > 0').reset_index(drop=True)

print(f"Nodes for rank-size analysis: {len(centrality_df)}")

# ----------------------------------------------------------
# 2. Rank-size helper
# ----------------------------------------------------------
def rank_size_data(values):
    """Sort descending, assign rank, return log-log arrays."""
    sorted_vals = np.sort(values)[::-1]
    ranks       = np.arange(1, len(sorted_vals) + 1)
    ln_rank     = np.log(ranks)
    ln_size     = np.log(sorted_vals)
    return ln_rank, ln_size

def fit_ols(ln_rank, ln_size):
    slope, intercept, r, p, _ = stats.linregress(ln_rank, ln_size)
    r2 = r ** 2
    return slope, intercept, r2

# ----------------------------------------------------------
# 3. Figure — 2×2 academic layout
# ----------------------------------------------------------
plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 10,
    'axes.linewidth'   : 0.8,
    'xtick.direction'  : 'out',
    'ytick.direction'  : 'out',
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
})

metrics = [
    ('degree',      'Degree weighted centrality'),
    ('closeness',   'Closeness weighted centrality'),
    ('pagerank',    'PageRank weighted centrality'),
    ('betweenness', 'Betweenness weighted centrality'),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 9), facecolor='white')
fig.patch.set_facecolor('white')

for ax, (col, title) in zip(axes.flatten(), metrics):

    vals              = centrality_df[col].values
    ln_rank, ln_size  = rank_size_data(vals)
    slope, intercept, r2 = fit_ols(ln_rank, ln_size)

    # fitted line range
    x_fit = np.linspace(ln_rank.min(), ln_rank.max(), 200)
    y_fit = slope * x_fit + intercept

    # scatter
    ax.scatter(ln_rank, ln_size,
               s=14, facecolors='none', edgecolors='#333333',
               linewidths=0.6, alpha=0.75, zorder=3)

    # OLS fit — dashed red
    ax.plot(x_fit, y_fit,
            color='#cc2222', linewidth=1.2,
            linestyle='--', zorder=4)

    # annotation
    eq_str  = (f'$R^2 = {r2:.2f}$\n'
               f'ln size $= {slope:.2f}$ ln rank '
               f'${"+" if intercept >= 0 else "-"}$ '
               f'${abs(intercept):.2f}$')
    ax.text(0.04, 0.08, eq_str,
            transform=ax.transAxes,
            fontsize=9, color='#111111',
            verticalalignment='bottom',
            bbox=dict(facecolor='white', edgecolor='none',
                      alpha=0.7, pad=2))

    ax.set_xlabel(f'ln {title} rank', fontsize=10)
    ax.set_ylabel(f'ln {title} size', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='normal',
                 color='#111111', pad=8)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_facecolor('white')
    ax.tick_params(labelsize=9)

plt.tight_layout(h_pad=3.5, w_pad=3)
plt.savefig('rank_size_centrality.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
BG = 'white'
fig, ax = plt.subplots(figsize=(6, 6), facecolor=BG)
ax.set_facecolor(BG)
ax.set_aspect('equal')
ax.axis('off')

# --- edges ---
for (u, v, data) in G_sub.edges(data=True):
    if u not in pos or v not in pos:
        continue
    w      = data.get('weight', 1)
    alpha  = 0.15 + 0.55 * (w / w_max)
    lw     = 0.2 + 3.0  * (w / w_max)   # width encodes intensity

    src_theme  = node_theme.get(u, 'Unknown')
    edge_color = THEME_COLOUR.get(src_theme, '#cccccc')

    x0, y0 = pos[u]
    x1, y1 = pos[v]

    ax.annotate(
        '', xy=(x1, y1), xytext=(x0, y0),
        arrowprops=dict(
            arrowstyle='-',
            color=edge_color,
            lw=lw,
            alpha=alpha,
            connectionstyle='arc3,rad=0.15',
        ),
        zorder=1,
    )

# --- nodes ---
for node in ordered_nodes:
    x, y      = pos[node]
    theme     = node_theme.get(node, 'Unknown')
    colour    = THEME_COLOUR.get(theme, '#aaaaaa')
    nw        = G_sub.nodes[node].get('weight', 1)
    node_size = 30 + 220 * (nw / w_node_max)

    ax.scatter(x, y, s=node_size, c=colour,
               edgecolors='#333333', linewidths=0.4,
               alpha=0.92, zorder=3)

# --- labels ---
for node in top_label_nodes:
    x, y  = pos[node]
    angle = np.degrees(np.arctan2(y, x))
    label = translate_poi(G_sub.nodes[node].get('label', str(node)))
    r     = 1.10
    lx, ly = r * x, r * y
    ha    = 'left' if x >= 0 else 'right'
    rotation = angle if -90 <= angle <= 90 else angle + 180

    ax.text(lx, ly, label,
            fontsize=7.5, color='#222222',
            ha=ha, va='center',
            rotation=rotation,
            rotation_mode='anchor',
            fontfamily='DejaVu Sans',
            zorder=4)

# --- theme arc labels ---
for theme, mid_a in theme_angle_map.items():
    r  = 1.32
    tx = r * np.cos(mid_a)
    ty = r * np.sin(mid_a)
    colour = THEME_COLOUR.get(theme, '#aaaaaa')
    ax.text(tx, ty, theme,
            fontsize=9, color=colour,
            ha='center', va='center',
            fontweight='bold',
            fontfamily='DejaVu Sans',
            zorder=5)

# --- legend ---
node_patches = [
    mpatches.Patch(facecolor=c, edgecolor='#333333',
                   linewidth=1, label=t)
    for t, c in THEME_COLOUR.items()
]
size_lines = [
    mpatches.Patch(facecolor='none', edgecolor='none',
                   label='Node size ∝ visit frequency'),
    mpatches.Patch(facecolor='none', edgecolor='none',
                   label='Edge width ∝ path frequency'),
]
legend = ax.legend(
    handles=node_patches + size_lines,
    loc='lower left',
    bbox_to_anchor=(0.01, 0.01),
    frameon=True, framealpha=0.9,
    facecolor='white', edgecolor='#cccccc',
    fontsize=8.5, labelcolor='#111111',
    title='Theme', title_fontsize=9,
)
legend.get_title().set_color('#111111')
legend.get_title().set_fontweight('bold')

ax.set_title(
    f'Citywalk Network — Circular Layout  (top {TOP_N} nodes)',
    color='#111111', fontsize=13, fontweight='bold',
    pad=20, fontfamily='DejaVu Sans',
)

plt.tight_layout()
plt.savefig('circular_network_white.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import networkx as nx

G = nx.DiGraph()

# Step 1: 先加所有有属性的节点
valid_cluster_ids = set(final_nodes_df['cluster_id'].tolist())

for _, row in final_nodes_df.iterrows():
    G.add_node(
        row['cluster_id'],
        label    = row['location_POI'],
        theme    = row['Primary_Theme'],
        weight   = row['point_weight'],
        mean_P   = row['mean_P'],
        mean_A   = row['mean_A'],
        mean_D   = row['mean_D'],
        pos      = (row.geometry.x, row.geometry.y),
    )

# Step 2: 只加两端都在nodes里的边
skipped = 0
for _, row in final_lines_df.iterrows():
    if row['start'] in valid_cluster_ids and row['end'] in valid_cluster_ids:
        G.add_edge(
            row['start'], row['end'],
            weight = row['line_weight'],
            theme  = row['Theme'],
        )
    else:
        skipped += 1

print(f"Nodes : {G.number_of_nodes()}")
print(f"Edges : {G.number_of_edges()}")
print(f"Skipped edges (orphan endpoint): {skipped}")

G_un = G.to_undirected()

# 重跑metrics
degree_in    = dict(G.in_degree(weight='weight'))
degree_out   = dict(G.out_degree(weight='weight'))
degree_total = dict(G.degree(weight='weight'))
betweenness  = nx.betweenness_centrality(G, weight='weight', normalized=True)
closeness    = nx.closeness_centrality(G)
pagerank     = nx.pagerank(G, weight='weight')

metrics_df = pd.DataFrame({
    'cluster_id'  : list(G.nodes()),
    'label'       : [G.nodes[n]['label']  for n in G.nodes()],
    'theme'       : [G.nodes[n]['theme']  for n in G.nodes()],
    'point_weight': [G.nodes[n]['weight'] for n in G.nodes()],
    'in_degree'   : [degree_in[n]         for n in G.nodes()],
    'out_degree'  : [degree_out[n]        for n in G.nodes()],
    'total_degree': [degree_total[n]      for n in G.nodes()],
    'betweenness' : [betweenness[n]       for n in G.nodes()],
    'closeness'   : [closeness[n]         for n in G.nodes()],
    'pagerank'    : [pagerank[n]          for n in G.nodes()],
}).sort_values('betweenness', ascending=False).reset_index(drop=True)

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
print(metrics_df.head())

In [ ]:
first_node = list(G.nodes())[0]
print(G.nodes[first_node])

In [ ]:
# ----------------------------------------------------------
# 2. Compute node-level metrics
# ----------------------------------------------------------
degree_in   = dict(G.in_degree(weight='weight'))
degree_out  = dict(G.out_degree(weight='weight'))
degree_total= dict(G.degree(weight='weight'))
betweenness = nx.betweenness_centrality(G, weight='weight', normalized=True)
closeness   = nx.closeness_centrality(G)
pagerank    = nx.pagerank(G, weight='weight')

metrics_df = pd.DataFrame({
    'cluster_id'  : list(G.nodes()),
    'label'       : [G.nodes[n]['label']  for n in G.nodes()],
    'theme'       : [G.nodes[n]['theme']  for n in G.nodes()],
    'point_weight': [G.nodes[n]['weight'] for n in G.nodes()],
    'in_degree'   : [degree_in[n]         for n in G.nodes()],
    'out_degree'  : [degree_out[n]        for n in G.nodes()],
    'total_degree': [degree_total[n]      for n in G.nodes()],
    'betweenness' : [betweenness[n]       for n in G.nodes()],
    'closeness'   : [closeness[n]         for n in G.nodes()],
    'pagerank'    : [pagerank[n]          for n in G.nodes()],
}).sort_values('betweenness', ascending=False).reset_index(drop=True)

print("\nTop 10 nodes by betweenness centrality:")
print(metrics_df[['label','theme','betweenness','pagerank','total_degree']].head(10).to_string())

In [ ]:
# ----------------------------------------------------------
# 3. Global network statistics
# ----------------------------------------------------------
components_weak  = nx.number_weakly_connected_components(G)
components_strong= nx.number_strongly_connected_components(G)

# Average clustering (undirected)
avg_clustering = nx.average_clustering(G_un)

# Degree assortativity
assortativity = nx.degree_assortativity_coefficient(G)

print(f"\nWeakly connected components  : {components_weak}")
print(f"Strongly connected components: {components_strong}")
print(f"Average clustering coefficient: {avg_clustering:.4f}")
print(f"Degree assortativity          : {assortativity:.4f}")

# Largest weakly connected component
largest_wcc = max(nx.weakly_connected_components(G), key=len)
G_lcc = G.subgraph(largest_wcc).copy()
print(f"\nLargest WCC — nodes: {G_lcc.number_of_nodes()}, edges: {G_lcc.number_of_edges()}")

In [ ]:
# ----------------------------------------------------------
# 4. Visualisation — 4-panel exploratory figure
# ----------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 11), facecolor='white')
fig.suptitle('Network Exploratory Analysis', fontsize=13,
             fontweight='bold', color='#111111', y=1.01)

THEME_COLOURS = {
    'Nature'               : '#7ed81a',
    'Local_Life'           : '#ff3d00',
    'Historical & Art'     : '#ff3fa4',
    'Commericial & Tourism': '#00ccff',
}

# ── (A) Degree distribution ───────────────────────────────
ax = axes[0, 0]
ax.set_facecolor('#fafafa')

degree_vals = [d for _, d in G.degree()]
ax.hist(degree_vals, bins=30, color='#4a90d9',
        edgecolor='white', linewidth=0.4)
ax.set_xlabel('Node degree', fontsize=10)
ax.set_ylabel('Count', fontsize=10)
ax.set_title('(A)  Degree Distribution', fontsize=11,
             fontweight='bold', loc='left', pad=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, color='#dddddd', linewidth=0.6, linestyle='--')
ax.set_axisbelow(True)

# ── (B) Top 15 nodes — betweenness centrality ────────────
ax = axes[0, 1]
ax.set_facecolor('#fafafa')

top15 = metrics_df.head(15)
colours_b = [THEME_COLOURS.get(t, '#aaaaaa') for t in top15['theme']]
bars = ax.barh(top15['label'][::-1], top15['betweenness'][::-1],
               color=colours_b[::-1], height=0.6,
               edgecolor='white', linewidth=0.4)
ax.set_xlabel('Betweenness centrality', fontsize=10)
ax.set_title('(B)  Top 15 Nodes — Betweenness Centrality',
             fontsize=11, fontweight='bold', loc='left', pad=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.xaxis.grid(True, color='#dddddd', linewidth=0.6, linestyle='--')
ax.set_axisbelow(True)
ax.tick_params(axis='y', labelsize=8, length=0)

legend_patches = [mpatches.Patch(facecolor=c, label=t)
                  for t, c in THEME_COLOURS.items()]
ax.legend(handles=legend_patches, fontsize=7.5, frameon=False,
          loc='lower right')

# ── (C) PageRank vs point_weight scatter ─────────────────
ax = axes[1, 0]
ax.set_facecolor('#fafafa')

sc_colours = [THEME_COLOURS.get(t, '#aaaaaa') for t in metrics_df['theme']]
ax.scatter(
    metrics_df['point_weight'],
    metrics_df['pagerank'],
    c=sc_colours, s=30, alpha=0.7, edgecolors='none'
)
ax.set_xlabel('Node visit frequency (point weight)', fontsize=10)
ax.set_ylabel('PageRank', fontsize=10)
ax.set_title('(C)  PageRank vs Visit Frequency',
             fontsize=11, fontweight='bold', loc='left', pad=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, color='#dddddd', linewidth=0.6, linestyle='--')
ax.xaxis.grid(True, color='#dddddd', linewidth=0.6, linestyle='--')
ax.set_axisbelow(True)

legend_patches = [mpatches.Patch(facecolor=c, label=t)
                  for t, c in THEME_COLOURS.items()]
ax.legend(handles=legend_patches, fontsize=7.5, frameon=False)

# ── (D) In-degree vs Out-degree scatter ──────────────────
ax = axes[1, 1]
ax.set_facecolor('#fafafa')

ax.scatter(
    metrics_df['in_degree'],
    metrics_df['out_degree'],
    c=sc_colours, s=30, alpha=0.7, edgecolors='none'
)

# Diagonal reference line (in = out)
max_deg = max(metrics_df['in_degree'].max(), metrics_df['out_degree'].max())
ax.plot([0, max_deg], [0, max_deg], color='#aaaaaa',
        linewidth=0.8, linestyle='--', label='in = out')

# Annotate top 5 by betweenness
for _, row in metrics_df.head(5).iterrows():
    ax.annotate(
        row['label'],
        xy=(row['in_degree'], row['out_degree']),
        xytext=(6, 4), textcoords='offset points',
        fontsize=7, color='#333333',
    )

ax.set_xlabel('Weighted in-degree', fontsize=10)
ax.set_ylabel('Weighted out-degree', fontsize=10)
ax.set_title('(D)  In-degree vs Out-degree',
             fontsize=11, fontweight='bold', loc='left', pad=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, color='#dddddd', linewidth=0.6, linestyle='--')
ax.xaxis.grid(True, color='#dddddd', linewidth=0.6, linestyle='--')
ax.set_axisbelow(True)
ax.legend(fontsize=8, frameon=False)

plt.tight_layout()
plt.savefig('fig_network_analysis.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# ============================================================
# TABLE — Centrality metrics for top 10 spots
# ============================================================

# All-node rankings (ascending rank = higher value = rank 1)
def rank_series(s):
    return s.rank(ascending=False, method='min').astype(int)

metrics_df['rank_betweenness'] = rank_series(metrics_df['betweenness'])
metrics_df['rank_closeness']   = rank_series(metrics_df['closeness'])
metrics_df['rank_pagerank']    = rank_series(metrics_df['pagerank'])
metrics_df['rank_in_degree']   = rank_series(metrics_df['in_degree'])
metrics_df['rank_out_degree']  = rank_series(metrics_df['out_degree'])
metrics_df['rank_total_degree']= rank_series(metrics_df['total_degree'])

# Filter to top10 spots by label
top10_labels = top10['location_POI'].tolist()
top10_metrics = metrics_df[metrics_df['label'].isin(top10_labels)].copy()

# Format each metric as "0.xxx (rank N)"
def fmt(val, rank):
    return f"{val:.3f}  (rank {rank})"

top10_metrics['poi_en'] = top10_metrics['label'].apply(translate_poi)

display_df = pd.DataFrame({
    'POI'            : top10_metrics['poi_en'].values,
    'Theme'          : top10_metrics['theme'].values,
    'In-degree'      : [fmt(v, r) for v, r in zip(top10_metrics['in_degree'],    top10_metrics['rank_in_degree'])],
    'Out-degree'     : [fmt(v, r) for v, r in zip(top10_metrics['out_degree'],   top10_metrics['rank_out_degree'])],
    'Total degree'   : [fmt(v, r) for v, r in zip(top10_metrics['total_degree'], top10_metrics['rank_total_degree'])],
    'Betweenness'    : [fmt(v, r) for v, r in zip(top10_metrics['betweenness'],  top10_metrics['rank_betweenness'])],
    'Closeness'      : [fmt(v, r) for v, r in zip(top10_metrics['closeness'],    top10_metrics['rank_closeness'])],
    'PageRank'       : [fmt(v, r) for v, r in zip(top10_metrics['pagerank'],     top10_metrics['rank_pagerank'])],
})

# Sort to match original top10 order
display_df = display_df.set_index('POI').loc[
    [translate_poi(l) for l in top10_labels]
].reset_index()

print(display_df.to_string(index=False))
display_df

## Network community + PAD

In [ ]:
# ============================================================
# COMMUNITY DETECTION — Emotion-based clustering
# ============================================================

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.spatial.distance import cdist

# ----------------------------------------------------------
# 1. Prepare PAD feature matrix (nodes with valid PAD only)
# ----------------------------------------------------------
pad_nodes = metrics_df.copy()
pad_nodes = pad_nodes.merge(
    final_nodes_df[['cluster_id', 'mean_P', 'mean_A', 'mean_D']],
    on='cluster_id', how='left'
)

valid_mask = pad_nodes[['mean_P', 'mean_A', 'mean_D']].notna().all(axis=1)
pad_valid  = pad_nodes[valid_mask].copy().reset_index(drop=True)

X = pad_valid[['mean_P', 'mean_A', 'mean_D']].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Nodes with valid PAD: {len(pad_valid)} / {len(pad_nodes)}")

# ----------------------------------------------------------
# 2. Determine optimal k via elbow + silhouette
# ----------------------------------------------------------
k_range = range(2, 9)
inertias, silhouettes = [], []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig_elbow, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), facecolor='white')
fig_elbow.suptitle('Optimal k Selection', fontsize=12,
                   fontweight='bold', color='#111111')

ax1.set_facecolor('#fafafa')
ax1.plot(list(k_range), inertias, 'o-', color='#4a90d9',
         linewidth=1.5, markersize=5)
ax1.set_xlabel('Number of clusters k', fontsize=10)
ax1.set_ylabel('Inertia', fontsize=10)
ax1.set_title('Elbow Method', fontsize=10, loc='left')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.xaxis.grid(True, color='#dddddd', linewidth=0.5, linestyle='--')
ax1.set_axisbelow(True)

ax2.set_facecolor('#fafafa')
ax2.plot(list(k_range), silhouettes, 'o-', color='#e05c2a',
         linewidth=1.5, markersize=5)
ax2.set_xlabel('Number of clusters k', fontsize=10)
ax2.set_ylabel('Silhouette score', fontsize=10)
ax2.set_title('Silhouette Score', fontsize=10, loc='left')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.xaxis.grid(True, color='#dddddd', linewidth=0.5, linestyle='--')
ax2.set_axisbelow(True)

plt.tight_layout()
plt.savefig('fig_elbow.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

best_k = int(input("Enter chosen k based on plots: ") or 3)

In [ ]:
# ----------------------------------------------------------
# 3. Final KMeans clustering
# ----------------------------------------------------------
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
pad_valid['emotion_cluster'] = km_final.fit_predict(X_scaled)

# Cluster centres in original PAD space
centres = scaler.inverse_transform(km_final.cluster_centers_)
centres_df = pd.DataFrame(centres, columns=['mean_P', 'mean_A', 'mean_D'])
centres_df.index.name = 'cluster'

# Auto-label clusters based on P and A
def label_cluster(row):
    p, a = row['mean_P'], row['mean_A']
    if p > 0.6 and a > 0.1:
        return 'High Pleasure / Active'
    elif p > 0.6 and a <= 0.1:
        return 'High Pleasure / Calm'
    elif p <= 0.6 and a > 0.1:
        return 'Moderate / Active'
    else:
        return 'Moderate / Relaxed'

centres_df['label'] = centres_df.apply(label_cluster, axis=1)
print("\nCluster centres:")
print(centres_df.round(3))

# Merge cluster back to pad_valid
pad_valid['cluster_label'] = pad_valid['emotion_cluster'].map(
    centres_df['label']
)

print("\nCluster size:")
print(pad_valid['emotion_cluster'].value_counts().sort_index())

In [ ]:
# ----------------------------------------------------------
# 4. Visualisation — 3-panel
# ----------------------------------------------------------
CLUSTER_COLOURS = [
    '#e05c2a', '#4a90d9', '#2ecc71', '#9b59b6',
    '#f1c40f', '#1abc9c', '#e74c3c', '#34495e'
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='white')
fig.suptitle('Emotion-based Community Detection (PAD K-Means)',
             fontsize=12, fontweight='bold', color='#111111', y=1.02)

# ── (A) P vs A scatter coloured by cluster ───────────────
ax = axes[0]
ax.set_facecolor('#fafafa')

for cid in range(best_k):
    mask = pad_valid['emotion_cluster'] == cid
    ax.scatter(
        pad_valid.loc[mask, 'mean_P'],
        pad_valid.loc[mask, 'mean_A'],
        c=CLUSTER_COLOURS[cid], s=25, alpha=0.75,
        edgecolors='none',
        label=centres_df.loc[cid, 'label'],
    )
# Cluster centres
ax.scatter(centres[:, 0], centres[:, 1],
           c='black', s=80, marker='x', linewidths=1.5,
           zorder=5, label='Centroid')
ax.axhline(0, color='#aaaaaa', linewidth=0.7, linestyle='--')
ax.axvline(0, color='#aaaaaa', linewidth=0.7, linestyle='--')
ax.set_xlabel('Pleasure (P)', fontsize=10)
ax.set_ylabel('Arousal (A)', fontsize=10)
ax.set_title('(A)  P × A Space', fontsize=11,
             fontweight='bold', loc='left', pad=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=7.5, frameon=False)

# ── (B) P vs D scatter ───────────────────────────────────
ax = axes[1]
ax.set_facecolor('#fafafa')

for cid in range(best_k):
    mask = pad_valid['emotion_cluster'] == cid
    ax.scatter(
        pad_valid.loc[mask, 'mean_P'],
        pad_valid.loc[mask, 'mean_D'],
        c=CLUSTER_COLOURS[cid], s=25, alpha=0.75,
        edgecolors='none',
    )
ax.scatter(centres[:, 0], centres[:, 2],
           c='black', s=80, marker='x', linewidths=1.5, zorder=5)
ax.axhline(0, color='#aaaaaa', linewidth=0.7, linestyle='--')
ax.axvline(0, color='#aaaaaa', linewidth=0.7, linestyle='--')
ax.set_xlabel('Pleasure (P)', fontsize=10)
ax.set_ylabel('Dominance (D)', fontsize=10)
ax.set_title('(B)  P × D Space', fontsize=11,
             fontweight='bold', loc='left', pad=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── (C) Cluster composition by Primary_Theme ─────────────
ax = axes[2]
ax.set_facecolor('#fafafa')

THEME_COLOURS = {
    'Nature'               : '#7ed81a',
    'Local_Life'           : '#ff3d00',
    'Historical & Art'     : '#ff3fa4',
    'Commericial & Tourism': '#00ccff',
}

comp = pad_valid.groupby(['emotion_cluster', 'theme']).size().unstack(fill_value=0)
comp_pct = comp.div(comp.sum(axis=1), axis=0) * 100

bottom = np.zeros(best_k)
for theme in comp_pct.columns:
    vals = comp_pct[theme].values
    ax.bar(
        [centres_df.loc[i, 'label'] for i in range(best_k)],
        vals, bottom=bottom,
        color=THEME_COLOURS.get(theme, '#aaaaaa'),
        edgecolor='white', linewidth=0.4,
        label=theme, width=0.55,
    )
    bottom += vals

ax.set_ylabel('Theme composition (%)', fontsize=10)
ax.set_title('(C)  Theme Composition per Cluster',
             fontsize=11, fontweight='bold', loc='left', pad=8)
ax.set_xticklabels(
    [centres_df.loc[i, 'label'] for i in range(best_k)],
    rotation=15, ha='right', fontsize=8.5
)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, color='#dddddd', linewidth=0.5, linestyle='--')
ax.set_axisbelow(True)

legend_patches = [mpatches.Patch(facecolor=c, label=t)
                  for t, c in THEME_COLOURS.items()]
ax.legend(handles=legend_patches, fontsize=7.5,
          frameon=False, loc='upper right')

plt.tight_layout()
plt.savefig('fig_emotion_clusters.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# 把emotion_cluster合并回final_nodes_df
final_nodes_df = final_nodes_df.merge(
    pad_valid[['cluster_id', 'emotion_cluster', 'cluster_label']],
    on='cluster_id', how='left'
)

# 在Folium地图上用cluster颜色渲染
CLUSTER_COLOURS_HEX = {
    0: '#e05c2a', 1: '#4a90d9', 2: '#2ecc71',
    3: '#9b59b6', 4: '#f1c40f',
}

def visualize_emotion_network(nodes_gdf, links_gdf):
    m = folium.Map(
        location=[nodes_gdf.geometry.y.mean(), nodes_gdf.geometry.x.mean()],
        zoom_start=14, tiles='CartoDB dark_matter'
    )

    for _, row in links_gdf.iterrows():
        line_pts = [[p[1], p[0]] for p in list(row['geometry'].coords)]
        folium.PolyLine(
            locations=line_pts,
            color='#555555', weight=max(1, min(row['line_weight'] * 1.5, 8)),
            opacity=0.4,
        ).add_to(m)

    for _, row in nodes_gdf.iterrows():
        if pd.isna(row.get('emotion_cluster')):
            colour = '#aaaaaa'
            label  = 'No PAD data'
        else:
            colour = CLUSTER_COLOURS_HEX.get(int(row['emotion_cluster']), '#aaaaaa')
            label  = row['cluster_label']

        node_radius = max(2, min(2 + row['point_weight'] * 0.05, 10))
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=node_radius,
            color='white', weight=0.5,
            fill=True, fill_color=colour, fill_opacity=0.85,
            popup=(
                f"<b>{row['location_POI']}</b><br>"
                f"Emotion cluster: {label}<br>"
                f"P={row['mean_P']:.2f}, A={row['mean_A']:.2f}, D={row['mean_D']:.2f}"
            )
        ).add_to(m)
    return m

emotion_map = visualize_emotion_network(final_nodes_df, final_lines_df)
emotion_map

In [ ]:
# 合并centrality指标和emotion cluster
hub_analysis = metrics_df.merge(
    pad_valid[['cluster_id', 'emotion_cluster', 'cluster_label',
               'mean_P', 'mean_A', 'mean_D']],
    on='cluster_id', how='left'
)

# ── 图：betweenness × PAD，cluster着色 ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5), facecolor='white')
fig.suptitle('Network Hubs — Emotional Profile',
             fontsize=12, fontweight='bold', y=1.01)

pad_dims = [
    ('mean_P', 'Pleasure (P)'),
    ('mean_A', 'Arousal (A)'),
    ('mean_D', 'Dominance (D)'),
]

for ax, (pad_col, pad_label), panel in zip(axes, pad_dims, 'ABC'):
    ax.set_facecolor('#fafafa')
    valid = hub_analysis.dropna(subset=[pad_col, 'emotion_cluster'])
    colours = [CLUSTER_COLOURS_HEX.get(int(c), '#aaaaaa')
               for c in valid['emotion_cluster']]

    ax.scatter(valid[pad_col], valid['betweenness'],
               c=colours, s=20 + valid['point_weight'] * 0.003,
               alpha=0.7, edgecolors='none')

    # Annotate top 5 hubs
    top5 = valid.nlargest(5, 'betweenness')
    for _, r in top5.iterrows():
        ax.annotate(
            translate_poi(r['label']),
            xy=(r[pad_col], r['betweenness']),
            xytext=(5, 3), textcoords='offset points',
            fontsize=7, color='#333333',
        )

    ax.set_xlabel(pad_label, fontsize=10)
    ax.set_ylabel('Betweenness centrality' if panel == 'A' else '', fontsize=10)
    ax.set_title(f'({panel})  Betweenness × {pad_label}',
                 fontsize=10, fontweight='bold', loc='left', pad=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.xaxis.grid(True, color='#dddddd', linewidth=0.5, linestyle='--')
    ax.yaxis.grid(True, color='#dddddd', linewidth=0.5, linestyle='--')
    ax.set_axisbelow(True)

# Shared cluster legend
patches = [mpatches.Patch(color=CLUSTER_COLOURS_HEX[i],
                           label=centres_df.loc[i, 'label'])
           for i in range(best_k)]
axes[2].legend(handles=patches, fontsize=7.5,
               frameon=False, loc='upper right')

plt.tight_layout()
plt.savefig('fig_hub_emotion.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# 对每条itinerary，追踪它穿越的emotion cluster序列
cluster_lookup = pad_valid.set_index('cluster_id')['emotion_cluster'].to_dict()

transition_records = []
sorted_df = df.sort_values(['note_index', 'location_index'])

for note_id, group in sorted_df.groupby('note_index'):
    cluster_seq = [
        final_nodes_df.loc[
            final_nodes_df['cluster_id'] == cid, 'cluster_id'
        ].values
        for cid in group['cluster_id'].tolist()
        if cid in cluster_lookup
    ]
    emotion_seq = [cluster_lookup[cid]
                   for cid in group['cluster_id'].tolist()
                   if cid in cluster_lookup]

    # Count transitions between emotion clusters
    for i in range(len(emotion_seq) - 1):
        s, e = emotion_seq[i], emotion_seq[i + 1]
        if s != e:
            transition_records.append({
                'note_index' : note_id,
                'from_cluster': s,
                'to_cluster'  : e,
                'from_label'  : centres_df.loc[s, 'label'],
                'to_label'    : centres_df.loc[e, 'label'],
            })

transitions_df = pd.DataFrame(transition_records)
trans_counts = (
    transitions_df
    .groupby(['from_label', 'to_label'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

print("Top emotion transitions:")
print(trans_counts.head(10).to_string(index=False))

# ── Chord-style transition heatmap ──────────────────────
pivot = trans_counts.pivot(index='from_label', columns='to_label', values='count').fillna(0)

fig_t, ax = plt.subplots(figsize=(7, 5), facecolor='white')
ax.set_facecolor('white')

im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=25, ha='right', fontsize=8.5)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=8.5)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = int(pivot.values[i, j])
        if val > 0:
            ax.text(j, i, str(val), ha='center', va='center',
                    fontsize=8, color='#111111')

cbar = fig_t.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Transition count', fontsize=9)
cbar.outline.set_visible(False)

ax.set_title('(C)  Emotion Cluster Transition Matrix',
             fontsize=11, fontweight='bold', loc='left', pad=10)
ax.spines[:].set_visible(False)

plt.tight_layout()
plt.savefig('fig_transitions.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

### Community detection

In [ ]:
# ============================================================
# COMMUNITY DETECTION — Infomap via NetworkX
# ============================================================
!pip install infomap

import infomap
import networkx as nx
import folium
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm

# ----------------------------------------------------------
# 1. Run Infomap
# ----------------------------------------------------------
im = infomap.Infomap("--directed --two-level --seed 42")

# Map cluster_id to consecutive integers for infomap
node_list = list(G.nodes())
node_to_idx = {n: i for i, n in enumerate(node_list)}
idx_to_node = {i: n for n, i in node_to_idx.items()}

for u, v, data in G.edges(data=True):
    im.add_link(node_to_idx[u], node_to_idx[v], data.get('weight', 1))

im.run()

# Extract community assignments
community_map = {}
for node in im.tree:
    if node.is_leaf:
        original_node = idx_to_node[node.node_id]
        community_map[original_node] = node.module_id

print(f"Number of communities found: {len(set(community_map.values()))}")

# Merge back to nodes
final_nodes_df['infomap_community'] = final_nodes_df['cluster_id'].map(community_map)

community_sizes = final_nodes_df['infomap_community'].value_counts()
print("\nCommunity sizes:")
print(community_sizes.to_string())

In [ ]:
# ----------------------------------------------------------
# 2. Assign colours to communities
# ----------------------------------------------------------
n_communities = len(community_sizes)
cmap = cm.get_cmap('tab20', n_communities)
community_colour = {
    cid: '#{:02x}{:02x}{:02x}'.format(
        int(cmap(i)[0]*255),
        int(cmap(i)[1]*255),
        int(cmap(i)[2]*255)
    )
    for i, cid in enumerate(community_sizes.index)
}

# ----------------------------------------------------------
# 3. Static matplotlib figure — community profile
# ----------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='white')
fig.suptitle('Infomap Community Detection', fontsize=12,
             fontweight='bold', color='#111111', y=1.01)

# ── (A) Community size bar chart ─────────────────────────
ax = axes[0]
ax.set_facecolor('#fafafa')

top_communities = community_sizes.head(12)
colours_bar = [community_colour[c] for c in top_communities.index]

bars = ax.barh(
    [f'Community {c}' for c in top_communities.index[::-1]],
    top_communities.values[::-1],
    color=colours_bar[::-1],
    height=0.6, edgecolor='white', linewidth=0.4
)
for bar, val in zip(bars, top_communities.values[::-1]):
    ax.text(bar.get_width() + 0.3,
            bar.get_y() + bar.get_height() / 2,
            str(val), va='center', ha='left',
            fontsize=8.5, color='#333333')

ax.set_xlabel('Number of nodes', fontsize=10)
ax.set_title('(A)  Community Sizes (top 12)',
             fontsize=11, fontweight='bold', loc='left', pad=8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.xaxis.grid(True, color='#dddddd', linewidth=0.5, linestyle='--')
ax.set_axisbelow(True)
ax.tick_params(axis='y', labelsize=8.5, length=0)

# ── (B) Theme composition per community ──────────────────
ax = axes[1]
ax.set_facecolor('#fafafa')

top_c_ids = top_communities.index.tolist()
comp = (
    final_nodes_df[final_nodes_df['infomap_community'].isin(top_c_ids)]
    .groupby(['infomap_community', 'Primary_Theme'])
    .size()
    .unstack(fill_value=0)
)
comp_pct = comp.div(comp.sum(axis=1), axis=0) * 100

bottom = np.zeros(len(top_c_ids))
x_labels = [f'C{c}' for c in top_c_ids]

for theme in comp_pct.columns:
    vals = [comp_pct.loc[c, theme] if c in comp_pct.index else 0
            for c in top_c_ids]
    ax.bar(x_labels, vals, bottom=bottom,
           color=THEME_COLOURS.get(theme, '#aaaaaa'),
           edgecolor='white', linewidth=0.4,
           label=theme, width=0.6)
    bottom += np.array(vals)

ax.set_ylabel('Theme composition (%)', fontsize=10)
ax.set_title('(B)  Theme Composition per Community',
             fontsize=11, fontweight='bold', loc='left', pad=8)
ax.set_xticklabels(x_labels, rotation=0, fontsize=8.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.yaxis.grid(True, color='#dddddd', linewidth=0.5, linestyle='--')
ax.set_axisbelow(True)

legend_patches = [mpatches.Patch(facecolor=c, label=t)
                  for t, c in THEME_COLOURS.items()]
ax.legend(handles=legend_patches, fontsize=7.5,
          frameon=False, loc='upper right')

plt.tight_layout()
plt.savefig('fig_infomap_profile.png', dpi=300,
            bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# ----------------------------------------------------------
# 4. Folium map — nodes coloured by infomap community
# ----------------------------------------------------------
theme_order = ['Nature', 'Commericial & Tourism', 'Local_Life', 'Historical & Art']
nodes_sorted = final_nodes_df.copy()
nodes_sorted['_order'] = nodes_sorted['Primary_Theme'].map(
    {t: i for i, t in enumerate(theme_order)}
).fillna(0)
nodes_sorted = nodes_sorted.sort_values('_order').drop(columns='_order')

m_im = folium.Map(
    location=[final_nodes_df.geometry.y.mean(),
              final_nodes_df.geometry.x.mean()],
    zoom_start=14,
    tiles='CartoDB dark_matter'
)

# Lines (grey, subdued)
for _, row in final_lines_df.iterrows():
    line_pts = [[p[1], p[0]] for p in list(row['geometry'].coords)]
    folium.PolyLine(
        locations=line_pts,
        color='#555555',
        weight=max(1, min(row['line_weight'] * 1.2, 6)),
        opacity=0.35,
    ).add_to(m_im)

# Nodes coloured by community
for _, row in nodes_sorted.iterrows():
    cid = row.get('infomap_community')
    colour = community_colour.get(cid, '#aaaaaa') if pd.notna(cid) else '#aaaaaa'
    node_radius = max(2, min(2 + row['point_weight'] * 0.05, 10))

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=node_radius,
        color='white', weight=0.4,
        fill=True, fill_color=colour, fill_opacity=0.85,
        popup=(
            f"<b>{row['location_POI']}</b><br>"
            f"Community: {cid}<br>"
            f"Theme: {row['Primary_Theme']}<br>"
            f"Visits: {row['point_weight']}"
        ),
        tooltip=f"{row['location_POI']} — C{cid}"
    ).add_to(m_im)

m_im

# Coupling Network analysis

In [ ]:
# ! pip install pypinyin
from pypinyin import lazy_pinyin, Style

POI_TRANSLATION_lines = {
    # === 自然与景区类 (Nature & Attractions) ===
    "植物园": "Xiamen Botanical Garden",      # 加城市限定词，避免全球歧义
    "日光岩": "Sunlight Rock",                # 鼓浪屿地标标准译名 (官方)
    "菽庄花园": "Shuzhuang Garden",           # 专名拼音 + 通名英文
    "南普陀寺": "Nanputuo Temple",            # 宗教场所标准译法
    "钟鼓索道": "Zhonggu Cable Car",          # 交通设施：拼音专名 + 功能通名
    "十里长堤": "Shili Causeway",             # 地理术语：Causeway 已含"长堤"义
    "龙舟池": "Dragon Boat Pool",             # 功能性地标直译
    # === 教育与文化类 (Education & Culture) ===
    "厦门大学": "Xiamen University",          # 官方英文校名
    "集美学村": "Jimei School Village",       # 历史街区固定译名
    "嘉庚公园": "Jiageng Park",               # 人名地名：拼音 + Park
    # === 城市公共空间 (Public Spaces) ===
    "中山公园": "Zhongshan Park",             # 标准公园译法
    "音乐广场": "Music Square",               # 标准广场译法
    "溪头下": "Xitouxia",                     # 村落/片区名：纯拼音保留原真性
    # === 道路与街巷 (Roads & Streets) ===
    "华新路": "Huaxin Road",                  # 拼音专名 + Road (DB3502 规范)
    "幸福路": "Xingfu Road",                  # 同上
    "公园西路": "Gongyuan West Road",         # 方位词标准译法：West + Road
    "湖明路":"Huming Road"
}

def to_pinyin(text):
    """Convert Chinese characters to title-cased pinyin romanisation."""
    try:
        syllables = lazy_pinyin(text, style=Style.NORMAL)
        return ' '.join(s.capitalize() for s in syllables)
    except:
        return text

def translate_poi(name):
    """Manual dict first, fallback to pinyin."""
    if name in POI_TRANSLATION:
        return POI_TRANSLATION[name]
    # check if string contains Chinese characters
    if any('\u4e00' <= c <= '\u9fff' for c in str(name)):
        return to_pinyin(name)
    return name

In [ ]:
metrics_df['label']==''

In [ ]:
plot3d

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler

# --- 假设前面的数据处理逻辑已经完成 ---
plot3d = metrics_df.merge(
    final_nodes_df[['cluster_id', 'mean_A']],
    on='cluster_id', how='left'
)

plot3d['poi_en'] = plot3d['label'].apply(translate_poi)

# 频次使用对数平滑，Betweenness 保持原始线性值
plot3d['log_weight'] = np.log1p(plot3d['point_weight'])
# 删除了原来有误导性的 plot3d['log_betweenness'] = plot3d['betweenness']

MANUAL_TRANSLATIONS = {
    "Hai Wan Gong Yuan": "Haiwan Park",
    "Bai Lu Zhou Gong Yuan": "Bailuzhou Park",
    "Zhong Hua Cheng": "Zhonghua City Bussiness Center",
    "Ying Guo Ling Shi Gong Guan Jiu Zhi": "Former British Consulate",
    "Tui Hua Gong Yuan": "Devolution Park",
}

# 使用字典更新英文列，如果在字典里找不到，就保留原样
plot3d['poi_en'] = plot3d['poi_en'].apply(lambda x: MANUAL_TRANSLATIONS.get(x, x))


# ==========================================
# 1. 提取候选标注点并进行空间过滤
# ==========================================
# 获取候选索引（综合最高频、最高介数、最高/低唤醒度）
candidate_idx = list(set(
    plot3d.nlargest(3, 'point_weight').index.tolist() +
    plot3d.nlargest(5, 'betweenness').index.tolist()  +
    plot3d.nlargest(3, 'mean_A').index.tolist()       +
    plot3d.nsmallest(3, 'mean_A').index.tolist()
))

# 提取候选点坐标及权重 (将原来的 log_betweenness 替换为真实的 betweenness)
candidates = plot3d.loc[candidate_idx, ['mean_A', 'betweenness', 'log_weight', 'point_weight']].copy()

# 三维坐标归一化（统一量纲，确保空间过滤时距离计算的准确性）
scaler = MinMaxScaler()
coords_scaled = scaler.fit_transform(candidates[['mean_A', 'betweenness', 'log_weight']])
candidates['x_s'] = coords_scaled[:, 0]
candidates['y_s'] = coords_scaled[:, 1]
candidates['z_s'] = coords_scaled[:, 2]

# 按权重从大到小排序（重叠时，优先保留权重大的点）
candidates = candidates.sort_values(by='point_weight', ascending=False)

final_label_indices = []
kept_coords = []

# === 阈值调整区 ===
# 0.05 ~ 0.15 之间微调。越大过滤得越狠，越小允许靠得越近。
DISTANCE_THRESHOLD = 0.15
# ================

for i, row in candidates.iterrows():
    current_coord = np.array([row['x_s'], row['y_s'], row['z_s']])

    is_too_close = False
    for kept_c in kept_coords:
        dist = np.linalg.norm(current_coord - kept_c)
        if dist < DISTANCE_THRESHOLD:
            is_too_close = True
            break

    if not is_too_close:
        final_label_indices.append(i)
        kept_coords.append(current_coord)

# 生成最终的标签 DataFrame
label_df = plot3d.loc[final_label_indices]

# ==========================================
# 2. 开始绘图
# ==========================================
fig = go.Figure()

# --- 主散点图 (按主题分类) ---
for theme, colour in THEME_COLOURS.items():
    mask = plot3d['theme'] == theme
    sub  = plot3d[mask]

    fig.add_trace(go.Scatter3d(
        x=sub['mean_A'],            # X轴: Arousal
        y=sub['betweenness'],       # Y轴: Betweenness (已修改为真实的线性列)
        z=sub['log_weight'],        # Z轴: Frequency
        mode='markers',
        name=theme,
        marker=dict(
            size=sub['point_weight'].apply(
                lambda w: max(6, min(w / 30, 12))
            ),
            color=colour,
            opacity=0.70,
            line=dict(width=0),
        ),
        text=sub['poi_en'],
        customdata=sub[['label', 'point_weight',
                         'betweenness', 'mean_A', 'theme']].values,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Visit frequency : %{customdata[1]}<br>"
            "Betweenness     : %{customdata[2]:.4f}<br>"
            "Arousal (A)     : %{customdata[3]:.3f}<br>"
            "Theme           : %{customdata[4]}"
            "<extra></extra>"
        ),
    ))

# --- 标签文本层 ---
fig.add_trace(go.Scatter3d(
    x=label_df['mean_A'],            # X轴同步
    y=label_df['betweenness'],       # Y轴同步 (已修改)
    z=label_df['log_weight'],        # Z轴同步
    mode='text',
    name='Labels',
    text=label_df['poi_en'],
    textfont=dict(
        size=10,
        color='#111111'
    ),
    textposition='top center',
    hoverinfo='skip',
    showlegend=False,
))

# --- 仅保留 Z轴 (Frequency) 的自定义对数刻度标签 ---
freq_ticks_raw  = [1, 10, 50, 100, 500, 1000, 5000]
freq_tick_vals  = [np.log1p(v) for v in freq_ticks_raw]
freq_tick_texts = [str(v) for v in freq_ticks_raw]

# (删除了 Betweenness 的自定义对数刻度配置，让 Plotly 自动生成线性刻度)

# --- 布局更新 ---
fig.update_layout(
    title=dict(
        text='Arousal × Betweenness Centrality × Visit Frequency',
        font=dict(size=14, color='#111111'),
        x=0.05,
    ),
    scene=dict(
        xaxis=dict(
            title='Arousal (A)',
            backgroundcolor='#f9f9f9',
            gridcolor='#dddddd',
            showbackground=True,
        ),
        yaxis=dict(
            title='Betweenness Centrality', # 去掉了 '(log)' 后缀
            # 去掉了 yaxis 这里的 tickvals 和 ticktext
            backgroundcolor='#f9f9f9',
            gridcolor='#dddddd',
            showbackground=True,
        ),
        zaxis=dict(
            title='Featured Frequency (log)',
            tickvals=freq_tick_vals,
            ticktext=freq_tick_texts,
            backgroundcolor='#f9f9f9',
            gridcolor='#dddddd',
            showbackground=True,
        ),
        camera=dict(eye=dict(x=1.6, y=1.6, z=0.8)),
    ),
    legend=dict(
        title=dict(text='Primary Theme', font=dict(size=10)),
        font=dict(size=9),
        bordercolor='#cccccc',
        borderwidth=0.5,
    ),
    paper_bgcolor='white',
    margin=dict(l=0, r=0, t=50, b=0),
    width=950,
    height=720,
)

# ==========================================
# 3. 添加 Arousal = 0 的参考平面
# ==========================================
# 获取当前 Y 轴 (Betweenness) 和 Z 轴 (Frequency) 的数据边界 (已修改为真实的 betweenness)
y_min = plot3d['betweenness'].min()
y_max = plot3d['betweenness'].max()
z_min = plot3d['log_weight'].min()
z_max = plot3d['log_weight'].max()

# 为了让平面看起来更完整，可以在边界上稍微向外延伸一点（可选）
y_padding = (y_max - y_min) * 0.05
z_padding = (z_max - z_min) * 0.05

fig.add_trace(go.Mesh3d(
    x=[0, 0, 0, 0], # X 轴固定在 0
    y=[y_min - y_padding, y_max + y_padding, y_max + y_padding, y_min - y_padding],
    z=[z_min - z_padding, z_min - z_padding, z_max + z_padding, z_max + z_padding],
    # 定义两个三角形来拼成一个矩形平面
    i=[0, 0],
    j=[1, 2],
    k=[2, 3],
    color='lightgray', # 平面颜色
    opacity=0.3,       # 半透明度，不要太深以免遮挡数据点
    name='Arousal = 0',
    hoverinfo='skip',  # 鼠标悬停时忽略平面
    showlegend=False   # 不在图例中显示
))

fig.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np
from adjustText import adjust_text

# ==========================================
# 开始循环绘制三张学术风格散点图
# ==========================================
for x_col, y_col in plot_combinations:
    # 1. 初始化 GridSpec 布局结构
    fig = plt.figure(figsize=(9, 9))
    gs = gridspec.GridSpec(
        2, 2,
        width_ratios=[1, 5],
        height_ratios=[5, 1],
        wspace=0.05, hspace=0.05
    )

    ax_main = plt.subplot(gs[0, 1])
    ax_y_marg = plt.subplot(gs[0, 0], sharey=ax_main)
    ax_x_marg = plt.subplot(gs[1, 1], sharex=ax_main)

    # --- 2. 绘制主散点图 ---
    for theme, colour in THEME_COLOURS.items():
        sub = plot3d[plot3d['theme'] == theme]

        # 🌟 修改点：取消了 sizes 的动态计算，直接将 s 设为固定值 80
        ax_main.scatter(
            sub[x_col], sub[y_col],
            s=80,  # <--- 你可以修改这个数字来统一调整所有点的大小
            c=colour, alpha=0.6,
            edgecolors='white', linewidths=0.6, label=theme
        )

    # --- 3. 防重叠的文字标签绘制逻辑 ---
    texts = []
    for idx, row in label_df.iterrows():
        t = ax_main.text(
            row[x_col], row[y_col],
            f" {row['poi_en']}",
            fontsize=9, color='#111111'
        )
        texts.append(t)

    # 自动调整文字位置，避免重叠
    adjust_text(
        texts,
        ax=ax_main,
        arrowprops=dict(arrowstyle="-", color='gray', lw=0.5, alpha=0.7)
    )

    # --- 后续的辅助线与边缘图逻辑保持不变 ---
    # Arousal = 0 的参考线
    if x_col == 'mean_A':
        ax_main.axvline(x=0, color='gray', linestyle='--', linewidth=1, alpha=0.6)
    elif y_col == 'mean_A':
        ax_main.axhline(y=0, color='gray', linestyle='--', linewidth=1, alpha=0.6)

    # 绘制 Y 轴边缘分布图
    sns.histplot(
        y=plot3d[y_col], ax=ax_y_marg,
        color="#7ca1cc", edgecolor="white", bins=20, stat="density", alpha=0.8
    )
    sns.kdeplot(y=plot3d[y_col], ax=ax_y_marg, color="#df6a5b", linewidth=2)
    ax_y_marg.invert_xaxis()
    ax_y_marg.axis('off')

    # 绘制 X 轴边缘分布图
    sns.histplot(
        x=plot3d[x_col], ax=ax_x_marg,
        color="#7ca1cc", edgecolor="white", bins=20, stat="density", alpha=0.8
    )
    sns.kdeplot(x=plot3d[x_col], ax=ax_x_marg, color="#df6a5b", linewidth=2)
    ax_x_marg.invert_yaxis()
    ax_x_marg.axis('off')

    # 主图坐标轴美化与学术风格调整
    ax_main.set_xlabel(axis_titles[x_col], fontsize=13, fontweight='bold', labelpad=10)
    ax_main.set_ylabel(axis_titles[y_col], fontsize=13, fontweight='bold', labelpad=10)

    ax_main.spines['top'].set_visible(False)
    ax_main.spines['right'].set_visible(False)
    ax_main.spines['left'].set_linewidth(1.5)
    ax_main.spines['bottom'].set_linewidth(1.5)

    ax_main.tick_params(axis='both', which='major', width=1.5, length=6, labelsize=11)

    # 还原对数刻度显示
    if x_col == 'log_weight':
        ax_main.set_xticks(axis_tick_configs[x_col]['vals'])
        ax_main.set_xticklabels(axis_tick_configs[x_col]['texts'])
    if y_col == 'log_weight':
        ax_main.set_yticks(axis_tick_configs[y_col]['vals'])
        ax_main.set_yticklabels(axis_tick_configs[y_col]['texts'])

    # ax_main.legend(
    #     title='Primary Theme',
    #     frameon=False, fontsize=10, title_fontsize=11, loc='best'
    # )

    plt.setp(ax_main.get_yticklabels()[0], visible=False)
    plt.setp(ax_main.get_xticklabels()[0], visible=False)

    plt.show()

## P-A-D community

In [ ]:
# ============================================================
# PAD-WEIGHTED COMMUNITY DETECTION — 3 variants
# ============================================================
import networkx as nx
import networkx.algorithms.community as nx_comm  # Replaces python-louvain
import pandas as pd
import numpy as np
import folium
from folium import plugins

# ----------------------------------------------------------
# 1. Helper: build weighted graph for each PAD dimension
# ----------------------------------------------------------

def build_pad_weighted_graph(G, pad_dim='mean_P', invert=False):
    """
    Returns a new undirected graph where edge weight =
    average PAD score of the two endpoint nodes.

    pad_dim : 'mean_P' | 'mean_A' | 'mean_D'
    invert  : if True, weight = 1 - norm(value)
              so low-PAD nodes attract each other
    """
    G_w = nx.Graph()

    for n, data in G.nodes(data=True):
        G_w.add_node(n, **data)

    for u, v, edata in G.edges(data=True):
        val_u = G.nodes[u].get(pad_dim, np.nan)
        val_v = G.nodes[v].get(pad_dim, np.nan)

        if pd.isna(val_u) or pd.isna(val_v):
            pad_w = 0.01
        else:
            avg   = (val_u + val_v) / 2          # [-1, 1]
            pad_w = (avg + 1) / 2                # normalise to [0, 1]
            if invert:
                pad_w = 1 - pad_w
            pad_w = max(pad_w, 0.01)

        # multiply by original traversal frequency
        trav_w = edata.get('weight', 1)
        G_w.add_edge(u, v, weight=pad_w * trav_w)

    return G_w


# ----------------------------------------------------------
# 2. Run Native NetworkX Louvain for each PAD dimension
# ----------------------------------------------------------

pad_dims = {
    'Pleasure'  : ('mean_P', False),
    'Arousal'   : ('mean_A', False),
    'Dominance' : ('mean_D', False),
}

community_results = {}

for dim_label, (pad_col, invert) in pad_dims.items():
    G_pad = build_pad_weighted_graph(G, pad_dim=pad_col, invert=invert)

    # Run native NetworkX Louvain algorithm
    # Returns a list of sets, where each set contains the nodes for one community
    communities = nx_comm.louvain_communities(G_pad, weight='weight', seed=42)

    # Calculate modularity using native NetworkX
    modularity = nx_comm.modularity(G_pad, communities, weight='weight')
    n_comm = len(communities)

    # Convert list of sets back into a {node: community_id} dictionary
    # to maintain compatibility with the rest of your pipeline
    partition = {node: cid for cid, comm_nodes in enumerate(communities) for node in comm_nodes}

    community_results[dim_label] = {
        'partition' : partition,
        'n_comm'    : n_comm,
        'modularity': modularity,
    }

    print(f"{dim_label:12} → {n_comm} communities, "
          f"modularity = {modularity:.4f}")

In [ ]:
# ----------------------------------------------------------
# 3. Merge community labels back to final_nodes_df
# ----------------------------------------------------------

for dim_label in pad_dims:
    col = f'comm_{dim_label.lower()}'
    final_nodes_df[col] = final_nodes_df['cluster_id'].map(
        community_results[dim_label]['partition']
    )

print(final_nodes_df[['location_POI', 'Primary_Theme',
                        'comm_pleasure', 'comm_arousal',
                        'comm_dominance']].head(10))


In [ ]:
# ----------------------------------------------------------
# 4. Summary table per community per dimension
# ----------------------------------------------------------

for dim_label, pad_col in [('Pleasure','mean_P'),
                             ('Arousal', 'mean_A'),
                             ('Dominance','mean_D')]:
    col = f'comm_{dim_label.lower()}'
    summary = (
        final_nodes_df
        .groupby(col)
        .agg(
            n_nodes      = (col,       'count'),
            mean_pad     = (pad_col,   'mean'),
            top_theme    = ('Primary_Theme',
                            lambda x: x.value_counts().index[0]),
            top_poi      = ('location_POI',
                            lambda x: x.value_counts().index[0]),
        )
        .sort_values('mean_pad', ascending=False)
        .reset_index()
    )
    print(f"\n{'='*50}")
    print(f"  {dim_label} communities")
    print(f"{'='*50}")
    print(summary.to_string(index=False))

In [ ]:
final_nodes_df

In [ ]:
# ----------------------------------------------------------
# 5. Folium 地理可视化 (从 geometry 动态提取坐标)
# ----------------------------------------------------------
import matplotlib.colors as mcolors
import folium
import pandas as pd

# 1. 从 geometry 列提取所有点的 y(纬度) 和 x(经度) 计算地图中心
# 使用 .apply() 可以兼容 GeoDataFrame 和包含 Shapely 对象的普通 DataFrame
center_lat = final_nodes_df['geometry'].apply(lambda p: p.y).mean()
center_lon = final_nodes_df['geometry'].apply(lambda p: p.x).mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles='CartoDB positron')

# 使用 matplotlib 提取一组具有高区分度的颜色
base_colors = list(mcolors.TABLEAU_COLORS.values()) + list(mcolors.XKCD_COLORS.values())

# 遍历三个维度，为每个维度创建一个独立的地图图层
for dim_label in ['Pleasure', 'Arousal', 'Dominance']:
    col_name = f'comm_{dim_label.lower()}'

    # 默认只显示 Pleasure 图层，其他隐藏以便于观察
    show_layer = True if dim_label == 'Pleasure' else False
    fg = folium.FeatureGroup(name=f'{dim_label} Communities', show=show_layer)

    for idx, row in final_nodes_df.iterrows():
        # 获取该节点的社区 ID
        comm_id = int(row[col_name]) if pd.notna(row[col_name]) else 0
        color = base_colors[comm_id % len(base_colors)]

        # 悬停弹窗显示的信息
        popup_html = (
            f"<b>POI:</b> {row.get('location_POI', 'N/A')}<br>"
            f"<b>Theme:</b> {row.get('Primary_Theme', 'N/A')}<br>"
            f"<b>{dim_label} Community:</b> {comm_id}"
        )

        # 2. 核心修改：直接从当前的 geometry 提取纬度(y)和经度(x)
        geom_point = row['geometry']
        lat, lon = geom_point.y, geom_point.x

        folium.CircleMarker(
            location=[lat, lon],
            radius=6,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
            weight=1,
            popup=folium.Popup(popup_html, max_width=300)
        ).add_to(fg)

    fg.add_to(m)

# 添加图层控制器
folium.LayerControl(collapsed=False).add_to(m)

# 保存并显示地图
m.save('pad_communities_map.html')
print("地图已保存为 'pad_communities_map.html'。")
display(m) # 如果你在 Jupyter 中，直接渲染